# 08 — Robustness analysis

Demo mode only. R2–R7 use the source definitions and 12-aspect taxonomy; the eight-class follow-up is separately namespaced and never replaces the 12-class main analysis. The 12-class taxonomy is the primary analysis. The 8-class version is a supporting high-coverage analysis. The reduced country threshold is used only for the synthetic demonstration. The formal study threshold is 500 papers. Each experiment writes to its own `results/demo/robustness` subdirectory. No network calls or source-data writes occur. Future use requires synthetic or authorized researcher-provided row-level data.

SOURCE_LINEAGE_LITERAL: `robustness_supplement_first_batch.ipynb` cells 0–10; `robustness_supplement_second_batch.ipynb` cells 0–6; `main_analysis_strict_sample.ipynb` cells 10–19 (eight-class follow-up). Cell 19 renders eight-class demo figures into its own output directory.


In [ ]:
from __future__ import annotations

import re
import warnings
from pathlib import Path
from typing import Iterable, Sequence

import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy.stats import chi2_contingency
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)


import re
import warnings
from pathlib import Path
from typing import Sequence

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy.stats import chi2_contingency, kruskal, mannwhitneyu
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

import re
import warnings
from itertools import combinations
from pathlib import Path
from typing import Iterable, Sequence

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm
from scipy.spatial.distance import jensenshannon
from scipy.stats import chi2_contingency
from statsmodels.stats.multitest import multipletests
import statsmodels.formula.api as smf

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

from pathlib import Path
import sys
import yaml

ROOT = Path.cwd().resolve()
if not (ROOT / "src" / "project_paths.py").exists():
    ROOT = ROOT.parent
if not (ROOT / "src" / "project_paths.py").exists():
    raise FileNotFoundError("Run from the release root or its notebooks directory.")
sys.path.insert(0, str(ROOT))
from src.project_paths import project_path

with (ROOT / "config" / "config.example.yaml").open(encoding="utf-8") as fh:
    CONFIG = yaml.safe_load(fh)
MODE = CONFIG.get("runtime", {}).get("mode", "demo")
if MODE not in ('demo',):
    raise ValueError("This notebook requires runtime.mode in: demo")
ANALYSIS = CONFIG["analysis"]
if int(ANALYSIS["formal_country_min_papers"]) != 500:
    raise ValueError("The formal country threshold must remain 500.")
if int(ANALYSIS["large_team_cutoff"]) != 50:
    raise ValueError("The formal large-team cutoff must remain 50.")
if ANALYSIS["country_reference"] != "US" or ANALYSIS["gender_reference"] != "male":
    raise ValueError("Formal model references must remain US and male.")
WORK_DIR = project_path("data/work")
STRICT_SAMPLES = project_path("results/demo/strict/samples")


## First robustness batch

In [ ]:
def run_first_batch():
    # =========================================================
    # 0. Paths and constants
    # =========================================================

    OUT_MASTER = WORK_DIR
    OUT_ASPECT = WORK_DIR
    OUT_MODEL = WORK_DIR
    OUT_SAMPLES = STRICT_SAMPLES
    OUT_ROBUST = project_path("results/demo/robustness/first_batch")
    OUT_TABLES = OUT_ROBUST / "tables"
    OUT_SAMPLES_ROBUST = OUT_ROBUST / "samples"

    for d in [OUT_TABLES, OUT_SAMPLES_ROBUST]:
        d.mkdir(parents=True, exist_ok=True)

    ASPECT_CLASSES = [
        "Methodology",
        "Results",
        "Experiments and Evaluation",
        "Data and Datasets",
        "Figures and Tables",
        "Writing, Language and Terminology",
        "Theory and Models",
        "Comparison and Context",
        "Paper Assessment",
        "Novelty and Impact",
        "Validity, Rigor and Quality",
        "Limitations",
    ]

    JSD_COL = "mean_pairwise_jsd"
    COUNTRY_MIN_PAPERS = int(CONFIG["demo"]["country_min_papers"])  # DEMO ONLY; formal value remains 500
    COUNTRY_REFERENCE = ANALYSIS["country_reference"]
    GENDER_REFERENCE = ANALYSIS["gender_reference"]
    GENDER_CONF_COL = "first_author_gender_confident"
    STRICT_COUNTRY_COL = "first_author_country_group_n500_strict"

    # Main strict sample excludes papers with >50 authors.
    # The sample-robustness experiment removes this cutoff.
    LARGE_TEAM_CUTOFF = int(ANALYSIS["large_team_cutoff"])


    # =========================================================
    # 1. General utilities
    # =========================================================

    def save_table(df: pd.DataFrame, path: Path) -> None:
        path.parent.mkdir(parents=True, exist_ok=True)
        if path.suffix == ".parquet":
            df.to_parquet(path, index=False)
        elif path.suffix == ".csv":
            df.to_csv(path, index=False, encoding="utf-8-sig")
        else:
            raise ValueError(f"Unsupported output format: {path}")
        print(f"Saved: {path} | rows={len(df):,}, cols={df.shape[1]}")


    def read_required(path: Path) -> pd.DataFrame:
        if not path.exists():
            raise FileNotFoundError(f"Required file not found: {path}")
        print(f"Reading: {path}")
        return pd.read_parquet(path)


    def read_preferred(paths: Sequence[Path]) -> pd.DataFrame:
        for p in paths:
            if p.exists():
                return read_required(p)
        raise FileNotFoundError("None of these files exists:\n" + "\n".join(map(str, paths)))


    def clean_value(x, unknown: str = "Unknown") -> str:
        if pd.isna(x):
            return unknown
        x = str(x).strip()
        if x == "" or x.lower() in {"nan", "none", "null", "na", "<na>"}:
            return unknown
        return x


    def clean_gender(x) -> str:
        x = clean_value(x).lower()
        if x in {"male", "m"}:
            return "male"
        if x in {"female", "f", "woman", "women"}:
            return "female"
        return "Unknown"


    def safe_log1p(series: pd.Series) -> pd.Series:
        s = pd.to_numeric(series, errors="coerce")
        s = s.where(s >= 0)
        return np.log1p(s)


    def safe_log(series: pd.Series) -> pd.Series:
        s = pd.to_numeric(series, errors="coerce")
        s = s.where(s > 0)
        return np.log(s)


    def sample_count(df: pd.DataFrame, label: str) -> dict:
        row = {"sample": label}
        if "paper_code" in df.columns:
            row["n_papers"] = int(df["paper_code"].nunique())
        if "review_id" in df.columns:
            row["n_reviews"] = int(df["review_id"].nunique())
        elif "n_reviews" in df.columns:
            row["n_reviews"] = int(pd.to_numeric(df["n_reviews"], errors="coerce").fillna(0).sum())
        return row


    def first_existing_col(df: pd.DataFrame, candidates: Sequence[str]) -> str | None:
        for c in candidates:
            if c in df.columns:
                return c
        return None


    def prepare_subject_primary(df: pd.DataFrame) -> pd.DataFrame:
        out = df.copy()
        if "subject_primary" in out.columns:
            out["subject_primary"] = out["subject_primary"].apply(clean_value)
        elif "subject_codes_final" in out.columns:
            out["subject_primary"] = (
                out["subject_codes_final"]
                .fillna("Unknown")
                .astype(str)
                .str.split("|")
                .str[0]
                .str.strip()
                .replace({"": "Unknown", "nan": "Unknown", "None": "Unknown"})
            )
        else:
            out["subject_primary"] = "Unknown"
        return out


    def harmonize_paper(df: pd.DataFrame) -> pd.DataFrame:
        out = prepare_subject_primary(df.copy())
        for col in [
            "publication_year",
            "n_authors",
            "n_reviews",
            "n_reviews_valid_for_jsd",
            "aspect_total_count",
            "mean_pairwise_jsd",
            "centroid_jsd",
            "max_pairwise_jsd",
        ]:
            if col in out.columns:
                out[col] = pd.to_numeric(out[col], errors="coerce")
        if GENDER_CONF_COL in out.columns:
            out[GENDER_CONF_COL] = out[GENDER_CONF_COL].apply(clean_gender)
        else:
            out[GENDER_CONF_COL] = "Unknown"
        if "n_authors" in out.columns:
            out["log_n_authors"] = safe_log1p(out["n_authors"])
        if "aspect_total_count" in out.columns:
            out["log_aspect_total_count"] = safe_log1p(out["aspect_total_count"])
        return out


    def add_country_group(df: pd.DataFrame, src_col: str, out_col: str, min_papers: int = COUNTRY_MIN_PAPERS) -> pd.DataFrame:
        if src_col not in df.columns:
            raise KeyError(f"Missing source country column: {src_col}")
        out = df.copy()
        country = out[src_col].apply(clean_value)
        counts = country[country != "Unknown"].value_counts()
        keep = set(counts[counts >= min_papers].index)
        out[out_col] = country.map(lambda x: "Unknown" if x == "Unknown" else (x if x in keep else "Other"))
        group_table = pd.DataFrame({
            "source_country_col": src_col,
            "country": counts.index,
            "n_papers": counts.values,
            "kept_as_separate_group": [c in keep for c in counts.index],
        })
        save_table(group_table, OUT_TABLES / f"country_grouping__{out_col}.csv")
        return out


    def coerce_for_statsmodels(
        df: pd.DataFrame,
        numeric_cols: Sequence[str] | None = None,
        categorical_cols: Sequence[str] | None = None,
        group_col: str | None = None,
    ) -> pd.DataFrame:
        out = df.copy()
        numeric_cols = list(numeric_cols or [])
        categorical_cols = list(categorical_cols or [])

        for col in numeric_cols:
            if col in out.columns:
                out[col] = pd.to_numeric(out[col], errors="coerce").astype("float64")
        for col in categorical_cols:
            if col in out.columns:
                out[col] = out[col].astype("object").where(out[col].notna(), "Unknown").astype(str)
        if group_col and group_col in out.columns:
            out[group_col] = out[group_col].astype("object").astype(str)

        # Patsy/statsmodels cannot always interpret pandas extension dtypes such as Int64Dtype().
        for col in out.columns:
            if pd.api.types.is_extension_array_dtype(out[col].dtype):
                if pd.api.types.is_numeric_dtype(out[col].dtype):
                    out[col] = pd.to_numeric(out[col], errors="coerce").astype("float64")
                elif pd.api.types.is_bool_dtype(out[col].dtype):
                    out[col] = out[col].astype("float64")
                else:
                    out[col] = out[col].astype("object").where(out[col].notna(), "Unknown").astype(str)
        return out


    def aspect_ratio_col(cls: str) -> str:
        return f"ratio__{re.sub(r'[^A-Za-z0-9]+', '_', cls).strip('_')}"


    def align_reviews_to_papers(review_counts: pd.DataFrame, paper_df: pd.DataFrame, extra_cols: Sequence[str] | None = None) -> pd.DataFrame:
        extra_cols = list(extra_cols or [])
        paper_cols = [
            "paper_code",
            "publication_year",
            "subject_primary",
            "n_authors",
            "log_n_authors",
            "n_reviews",
            GENDER_CONF_COL,
            STRICT_COUNTRY_COL,
        ] + extra_cols
        paper_cols = [c for c in dict.fromkeys(paper_cols) if c in paper_df.columns]
        out = review_counts.copy()
        out = out[out["paper_code"].isin(paper_df["paper_code"])].copy()
        drop_cols = [c for c in paper_cols if c != "paper_code" and c in out.columns]
        out = out.drop(columns=drop_cols, errors="ignore")
        out = out.merge(paper_df[paper_cols], on="paper_code", how="left")
        for c in ASPECT_CLASSES + ["aspect_total_count"]:
            if c in out.columns:
                out[c] = pd.to_numeric(out[c], errors="coerce").fillna(0)
        if "review_word_count" in out.columns:
            out["log_review_word_count"] = safe_log1p(out["review_word_count"])
        else:
            out["log_review_word_count"] = 0.0
        return out


    # =========================================================
    # 2. Load main samples and source tables
    # =========================================================

    paper_main = read_required(OUT_SAMPLES / "main_strict_paper_jsd_no_large50.parquet")
    review_main = read_required(OUT_SAMPLES / "main_strict_review_aligned.parquet")

    paper_bg = read_preferred([
        OUT_MASTER / "paper_background_features_v2.parquet",
    ])
    review_counts = read_preferred([
        OUT_ASPECT / "review_aspect_class_counts_v2.parquet",
    ])
    paper_jsd_source = read_preferred([
        OUT_MODEL / "paper_aspect_disagreement_v2.parquet",
    ])

    paper_main = harmonize_paper(paper_main)
    review_main = review_main.copy()

    # Merge any missing background columns into the strict main sample for robustness checks.
    bg_merge_candidates = [
        "paper_code",
        "first_author_country",
        "corresponding_author_country",
        "dominant_country",
        "first_author_gender_confident",
        "first_author_name_inferred_gender",
        "first_author_gender_raw",
        "first_author_gender",
        "first_author_gender_probability",
        "first_author_gender_prob",
        "first_author_gender_count",
        "first_author_genderize_count",
        "last_author_gender_confident",
        "first_last_gender_pair",
        "publication_year",
        "subject_primary",
        "subject_codes_final",
        "n_authors",
    ]
    bg_cols = [c for c in bg_merge_candidates if c in paper_bg.columns]
    missing_cols = [c for c in bg_cols if c != "paper_code" and c not in paper_main.columns]
    if missing_cols:
        paper_main = paper_main.merge(paper_bg[["paper_code"] + missing_cols], on="paper_code", how="left")
        paper_main = harmonize_paper(paper_main)

    # Review counts used to build aligned review samples for relaxed/alternative analyses.
    review_counts = review_counts.copy()
    if "aspect_total_count" in review_counts.columns:
        review_counts = review_counts[pd.to_numeric(review_counts["aspect_total_count"], errors="coerce").fillna(0) > 0].copy()

    # =========================================================
    # 3. Shared model functions
    # =========================================================

    def fit_paper_jsd_ols(
        df: pd.DataFrame,
        y_col: str,
        predictor_type: str,
        formula_rhs: str,
        prefix: str,
        country_col: str | None = None,
        gender_col: str | None = None,
    ) -> pd.DataFrame:
        needed = [
            "paper_code",
            y_col,
            "publication_year",
            "subject_primary",
            "n_reviews_valid_for_jsd",
            "log_aspect_total_count",
            "log_n_authors",
        ]
        if country_col:
            needed.append(country_col)
        if gender_col:
            needed.append(gender_col)
        needed = [c for c in dict.fromkeys(needed) if c in df.columns]
        model_df = df[needed].dropna().copy()

        categorical_cols = ["subject_primary"]
        if country_col:
            categorical_cols.append(country_col)
        if gender_col:
            categorical_cols.append(gender_col)
        model_df = coerce_for_statsmodels(
            model_df,
            numeric_cols=[y_col, "publication_year", "n_reviews_valid_for_jsd", "log_aspect_total_count", "log_n_authors"],
            categorical_cols=categorical_cols,
            group_col="paper_code",
        )
        model_df = model_df.dropna().copy()
        if model_df.empty:
            return pd.DataFrame()

        formula = f"Q('{y_col}') ~ {formula_rhs}"
        model = smf.ols(formula, data=model_df).fit(cov_type="HC3")
        conf_int = model.conf_int()
        rows = []
        for term, est in model.params.items():
            if term == "Intercept":
                continue
            conf = conf_int.loc[term]
            rows.append({
                "dependent_var": y_col,
                "predictor_type": predictor_type,
                "term": term,
                "estimate": est,
                "conf_low": conf[0],
                "conf_high": conf[1],
                "p_value": model.pvalues.get(term, np.nan),
                "n_papers": int(model.nobs),
                "r_squared": model.rsquared,
                "adj_r_squared": model.rsquared_adj,
                "model": prefix,
            })
        out = pd.DataFrame(rows)
        if len(out) > 0:
            out["p_adj_bh"] = multipletests(out["p_value"].fillna(1), method="fdr_bh")[1]
        return out


    def build_aspect_contingency(df: pd.DataFrame, group_col: str, exclude_groups: Iterable[str] | None = None) -> pd.DataFrame:
        temp = df.copy()
        temp[group_col] = temp[group_col].apply(clean_value)
        if exclude_groups is not None:
            temp = temp[~temp[group_col].isin(set(exclude_groups))].copy()
        cont = temp.groupby(group_col)[ASPECT_CLASSES].sum().astype(float)
        cont = cont.loc[cont.sum(axis=1) > 0]
        return cont


    def cramers_v(chi2: float, n: float, r: int, c: int) -> float:
        denom = n * min(r - 1, c - 1)
        return float(np.sqrt(chi2 / denom)) if denom > 0 else np.nan


    def chi_square_aspect_test(df: pd.DataFrame, group_col: str, prefix: str, exclude_groups: Iterable[str] | None = None) -> pd.DataFrame:
        cont = build_aspect_contingency(df, group_col, exclude_groups=exclude_groups)
        chi2, p, dof, expected = chi2_contingency(cont.values)
        n = cont.values.sum()
        summary = pd.DataFrame([{
            "group_col": group_col,
            "n_groups": cont.shape[0],
            "n_aspects": cont.shape[1],
            "n_aspect_mentions": n,
            "chi2": chi2,
            "dof": dof,
            "p_value": p,
            "cramers_v": cramers_v(chi2, n, cont.shape[0], cont.shape[1]),
        }])
        save_table(summary, OUT_TABLES / f"test_chi_square_aspect__{prefix}.csv")
        cont.to_csv(OUT_TABLES / f"test_chi_square_aspect__{prefix}__observed.csv", encoding="utf-8-sig")
        pd.DataFrame(expected, index=cont.index, columns=cont.columns).to_csv(
            OUT_TABLES / f"test_chi_square_aspect__{prefix}__expected.csv", encoding="utf-8-sig"
        )
        return summary


    def add_review_model_features(df: pd.DataFrame) -> pd.DataFrame:
        out = df.copy()
        total = pd.to_numeric(out["aspect_total_count"], errors="coerce")
        for cls in ASPECT_CLASSES:
            out[cls] = pd.to_numeric(out[cls], errors="coerce").fillna(0)
            out[aspect_ratio_col(cls)] = np.where(total > 0, out[cls] / total, np.nan)
        out["log_aspect_total_count_offset"] = safe_log(total)
        if "log_review_word_count" not in out.columns:
            out["log_review_word_count"] = safe_log1p(out["review_word_count"] if "review_word_count" in out.columns else pd.Series(0, index=out.index))
        return out


    def fit_review_aspect_ols(
        df: pd.DataFrame,
        predictor_type: str,
        formula_rhs: str,
        prefix: str,
        country_col: str | None = None,
        gender_col: str | None = None,
    ) -> pd.DataFrame:
        temp = add_review_model_features(df)
        rows = []
        for cls in ASPECT_CLASSES:
            y = aspect_ratio_col(cls)
            needed = [
                y,
                "paper_code",
                "publication_year",
                "subject_primary",
                "log_review_word_count",
                "log_n_authors",
            ]
            if country_col:
                needed.append(country_col)
            if gender_col:
                needed.append(gender_col)
            needed = [c for c in dict.fromkeys(needed) if c in temp.columns]
            model_df = temp[needed].dropna().copy()
            categorical = ["subject_primary"]
            if country_col:
                categorical.append(country_col)
            if gender_col:
                categorical.append(gender_col)
            model_df = coerce_for_statsmodels(
                model_df,
                numeric_cols=[y, "publication_year", "log_review_word_count", "log_n_authors"],
                categorical_cols=categorical,
                group_col="paper_code",
            ).dropna()
            if model_df.empty:
                continue
            formula = f"Q('{y}') ~ {formula_rhs}"
            try:
                model = smf.ols(formula, data=model_df).fit(cov_type="cluster", cov_kwds={"groups": model_df["paper_code"]})
            except Exception as e:
                print(f"OLS review model failed: {cls} | {prefix} | {e}")
                continue
            conf_int = model.conf_int()
            for term, est in model.params.items():
                if term == "Intercept":
                    continue
                conf = conf_int.loc[term]
                rows.append({
                    "aspect_class": cls,
                    "predictor_type": predictor_type,
                    "term": term,
                    "estimate": est,
                    "conf_low": conf[0],
                    "conf_high": conf[1],
                    "p_value": model.pvalues.get(term, np.nan),
                    "n_reviews": int(model.nobs),
                    "n_papers": model_df["paper_code"].nunique(),
                    "model_type": "OLS_ratio",
                    "model": prefix,
                })
        out = pd.DataFrame(rows)
        if len(out) > 0:
            out["p_adj_bh"] = multipletests(out["p_value"].fillna(1), method="fdr_bh")[1]
        return out


    def fit_review_aspect_poisson_offset(
        df: pd.DataFrame,
        predictor_type: str,
        formula_rhs: str,
        prefix: str,
        country_col: str | None = None,
        gender_col: str | None = None,
    ) -> pd.DataFrame:
        temp = add_review_model_features(df)
        rows = []
        for cls in ASPECT_CLASSES:
            needed = [
                cls,
                "paper_code",
                "publication_year",
                "subject_primary",
                "log_review_word_count",
                "log_n_authors",
                "log_aspect_total_count_offset",
            ]
            if country_col:
                needed.append(country_col)
            if gender_col:
                needed.append(gender_col)
            needed = [c for c in dict.fromkeys(needed) if c in temp.columns]
            model_df = temp[needed].dropna().copy()
            categorical = ["subject_primary"]
            if country_col:
                categorical.append(country_col)
            if gender_col:
                categorical.append(gender_col)
            model_df = coerce_for_statsmodels(
                model_df,
                numeric_cols=[cls, "publication_year", "log_review_word_count", "log_n_authors", "log_aspect_total_count_offset"],
                categorical_cols=categorical,
                group_col="paper_code",
            ).dropna()
            if model_df.empty:
                continue
            formula = f"Q('{cls}') ~ {formula_rhs}"
            try:
                model = smf.glm(
                    formula=formula,
                    data=model_df,
                    family=sm.families.Poisson(),
                    offset=model_df["log_aspect_total_count_offset"],
                ).fit(cov_type="cluster", cov_kwds={"groups": model_df["paper_code"]})
            except Exception as e:
                print(f"Poisson review model failed: {cls} | {prefix} | {e}")
                continue
            conf_int = model.conf_int()
            for term, est in model.params.items():
                if term == "Intercept":
                    continue
                conf = conf_int.loc[term]
                rows.append({
                    "aspect_class": cls,
                    "predictor_type": predictor_type,
                    "term": term,
                    "estimate": est,
                    "conf_low": conf[0],
                    "conf_high": conf[1],
                    "p_value": model.pvalues.get(term, np.nan),
                    "n_reviews": int(model.nobs),
                    "n_papers": model_df["paper_code"].nunique(),
                    "model_type": "Poisson_offset",
                    "model": prefix,
                })
        out = pd.DataFrame(rows)
        if len(out) > 0:
            out["p_adj_bh"] = multipletests(out["p_value"].fillna(1), method="fdr_bh")[1]
        return out


    def compare_ols_poisson(ols: pd.DataFrame, pois: pd.DataFrame, prefix: str) -> pd.DataFrame:
        """Compare OLS ratio and Poisson-offset results."""
        keys = ["aspect_class", "predictor_type", "term"]

        if ols.empty or pois.empty:
            out = pd.DataFrame()
        else:
            o = ols.copy()
            p = pois.copy()

            o = o[[c for c in keys + [
                "estimate", "conf_low", "conf_high", "p_value", "p_adj_bh",
                "n_reviews", "n_papers", "model_type", "model"
            ] if c in o.columns]].copy()

            p = p[[c for c in keys + [
                "estimate", "conf_low", "conf_high", "p_value", "p_adj_bh",
                "n_reviews", "n_papers", "model_type", "model"
            ] if c in p.columns]].copy()

            o = o.rename(columns={
                "estimate": "estimate_ols",
                "conf_low": "conf_low_ols",
                "conf_high": "conf_high_ols",
                "p_value": "p_ols",
                "p_adj_bh": "p_adj_ols",
                "n_reviews": "n_reviews_ols",
                "n_papers": "n_papers_ols",
                "model_type": "model_type_ols",
                "model": "model_ols",
            })

            p = p.rename(columns={
                "estimate": "estimate_poisson",
                "conf_low": "conf_low_poisson",
                "conf_high": "conf_high_poisson",
                "p_value": "p_poisson",
                "p_adj_bh": "p_adj_poisson",
                "n_reviews": "n_reviews_poisson",
                "n_papers": "n_papers_poisson",
                "model_type": "model_type_poisson",
                "model": "model_poisson",
            })

            o = o.drop_duplicates(subset=keys)
            p = p.drop_duplicates(subset=keys)

            out = o.merge(p, on=keys, how="inner")

            out["same_direction"] = np.sign(out["estimate_ols"]) == np.sign(out["estimate_poisson"])
            out["both_p_lt_05"] = (out["p_ols"] < 0.05) & (out["p_poisson"] < 0.05)

            if "p_adj_ols" in out.columns and "p_adj_poisson" in out.columns:
                out["both_fdr_lt_05"] = (out["p_adj_ols"] < 0.05) & (out["p_adj_poisson"] < 0.05)
            else:
                out["both_fdr_lt_05"] = False

        save_table(out, OUT_TABLES / f"compare_review_ols_ratio_vs_poisson_offset__{prefix}.csv")

        if len(out) > 0:
            summary = pd.DataFrame([{
                "model": prefix,
                "n_terms_compared": len(out),
                "same_direction_n": int(out["same_direction"].sum()),
                "same_direction_share": float(out["same_direction"].mean()),
                "both_p_lt_05_n": int(out["both_p_lt_05"].sum()),
                "both_p_lt_05_share": float(out["both_p_lt_05"].mean()),
                "both_fdr_lt_05_n": int(out["both_fdr_lt_05"].sum()),
                "both_fdr_lt_05_share": float(out["both_fdr_lt_05"].mean()),
            }])
        else:
            summary = pd.DataFrame([{"model": prefix, "n_terms_compared": 0}])

        save_table(summary, OUT_TABLES / f"compare_review_ols_ratio_vs_poisson_offset__{prefix}__summary.csv")
        return out


    # =========================================================
    # 4. R2: sample robustness by including >50-author papers
    # =========================================================

    # Rebuild a relaxed paper sample from source tables, using the same logic as the strict sample but without n_authors <= 50.
    merge_cols = [
        "paper_code",
        "n_authors",
        "first_author_country",
        "corresponding_author_country",
        "dominant_country",
        GENDER_CONF_COL,
        "publication_year",
        "subject_primary",
        "subject_codes_final",
    ]
    merge_cols = [c for c in merge_cols if c in paper_bg.columns]
    missing_in_jsd = [c for c in merge_cols if c != "paper_code" and c not in paper_jsd_source.columns]
    paper_relaxed_source = paper_jsd_source.copy()
    if missing_in_jsd:
        paper_relaxed_source = paper_relaxed_source.merge(paper_bg[["paper_code"] + missing_in_jsd], on="paper_code", how="left")
    paper_relaxed_source = harmonize_paper(paper_relaxed_source)

    paper_relaxed = paper_relaxed_source[
        paper_relaxed_source[JSD_COL].notna()
        & (pd.to_numeric(paper_relaxed_source["n_reviews_valid_for_jsd"], errors="coerce") >= 2)
        & paper_relaxed_source["n_authors"].notna()
    ].copy()

    RELAXED_COUNTRY_COL = "first_author_country_group_n500_relaxed"
    paper_relaxed = add_country_group(paper_relaxed, "first_author_country", RELAXED_COUNTRY_COL)
    paper_relaxed = harmonize_paper(paper_relaxed)

    paper_relaxed_country = paper_relaxed[paper_relaxed[RELAXED_COUNTRY_COL] != "Unknown"].copy()
    paper_relaxed_gender = paper_relaxed[paper_relaxed[GENDER_CONF_COL].isin(["male", "female"])].copy()
    paper_relaxed_combined = paper_relaxed[(paper_relaxed[RELAXED_COUNTRY_COL] != "Unknown") & paper_relaxed[GENDER_CONF_COL].isin(["male", "female"])].copy()

    review_relaxed = align_reviews_to_papers(review_counts, paper_relaxed, extra_cols=[RELAXED_COUNTRY_COL])
    review_relaxed_country = review_relaxed[review_relaxed[RELAXED_COUNTRY_COL] != "Unknown"].copy()
    review_relaxed_gender = review_relaxed[review_relaxed[GENDER_CONF_COL].isin(["male", "female"])].copy()

    save_table(paper_relaxed, OUT_SAMPLES_ROBUST / "robust_relaxed_include_large_teams_paper.parquet")
    save_table(review_relaxed, OUT_SAMPLES_ROBUST / "robust_relaxed_include_large_teams_review.parquet")

    r2_samples = pd.DataFrame([
        sample_count(paper_main, "strict main paper sample: n_authors <= 50"),
        sample_count(review_main, "strict main review sample: n_authors <= 50"),
        sample_count(paper_relaxed, "relaxed paper sample: no author-count cutoff"),
        sample_count(review_relaxed, "relaxed review sample: no author-count cutoff"),
        sample_count(paper_relaxed_country, "relaxed country-known paper sample"),
        sample_count(review_relaxed_country, "relaxed country-known review sample"),
        sample_count(paper_relaxed_gender, "relaxed gender-known paper sample"),
        sample_count(review_relaxed_gender, "relaxed gender-known review sample"),
    ])
    save_table(r2_samples, OUT_TABLES / "R2_sample_robustness__sample_counts.csv")

    base_jsd_controls = "publication_year + C(subject_primary) + n_reviews_valid_for_jsd + log_aspect_total_count + log_n_authors"
    r2_country = fit_paper_jsd_ols(
        paper_relaxed_country,
        y_col=JSD_COL,
        predictor_type="first_author_country_relaxed_sample",
        formula_rhs=f"C({RELAXED_COUNTRY_COL}, Treatment(reference='{COUNTRY_REFERENCE}')) + {base_jsd_controls}",
        prefix="R2_relaxed_country",
        country_col=RELAXED_COUNTRY_COL,
    )
    r2_gender = fit_paper_jsd_ols(
        paper_relaxed_gender,
        y_col=JSD_COL,
        predictor_type="first_author_gender_relaxed_sample",
        formula_rhs=f"C({GENDER_CONF_COL}, Treatment(reference='{GENDER_REFERENCE}')) + {base_jsd_controls}",
        prefix="R2_relaxed_gender",
        gender_col=GENDER_CONF_COL,
    )
    r2_combined = fit_paper_jsd_ols(
        paper_relaxed_combined,
        y_col=JSD_COL,
        predictor_type="country_and_gender_relaxed_sample",
        formula_rhs=f"C({RELAXED_COUNTRY_COL}, Treatment(reference='{COUNTRY_REFERENCE}')) + C({GENDER_CONF_COL}, Treatment(reference='{GENDER_REFERENCE}')) + {base_jsd_controls}",
        prefix="R2_relaxed_country_gender",
        country_col=RELAXED_COUNTRY_COL,
        gender_col=GENDER_CONF_COL,
    )
    save_table(pd.concat([r2_country, r2_gender, r2_combined], ignore_index=True), OUT_TABLES / "R2_model_paper_jsd_ols__relaxed_include_large_teams.csv")
    chi_square_aspect_test(review_relaxed_country, RELAXED_COUNTRY_COL, "R2_relaxed_country", exclude_groups=["Unknown"])
    chi_square_aspect_test(review_relaxed_gender, GENDER_CONF_COL, "R2_relaxed_gender", exclude_groups=["Unknown"])

    # =========================================================
    # 5. R3: country attribution robustness
    # =========================================================

    r3_rows = []
    r3_models = []
    for src_col, label in [
        ("corresponding_author_country", "corresponding_author_country"),
        ("dominant_country", "dominant_country"),
    ]:
        if src_col not in paper_main.columns:
            print(f"Skipping R3 {src_col}: column not found.")
            continue
        out_col = f"{label}_group_n500_strict"
        p_alt = add_country_group(paper_main, src_col, out_col)
        p_alt = harmonize_paper(p_alt)
        p_alt_known = p_alt[p_alt[out_col] != "Unknown"].copy()
        r_alt = align_reviews_to_papers(review_counts, p_alt, extra_cols=[out_col])
        r_alt_known = r_alt[r_alt[out_col] != "Unknown"].copy()
        save_table(p_alt_known, OUT_SAMPLES_ROBUST / f"R3_{label}__paper_known.parquet")
        save_table(r_alt_known, OUT_SAMPLES_ROBUST / f"R3_{label}__review_known.parquet")
        r3_rows.extend([
            sample_count(p_alt_known, f"R3 paper sample: {label} known"),
            sample_count(r_alt_known, f"R3 review sample: {label} known"),
        ])
        m = fit_paper_jsd_ols(
            p_alt_known,
            y_col=JSD_COL,
            predictor_type=label,
            formula_rhs=f"C({out_col}, Treatment(reference='{COUNTRY_REFERENCE}')) + {base_jsd_controls}",
            prefix=f"R3_{label}",
            country_col=out_col,
        )
        r3_models.append(m)
        chi_square_aspect_test(r_alt_known, out_col, f"R3_{label}", exclude_groups=["Unknown"])

    save_table(pd.DataFrame(r3_rows), OUT_TABLES / "R3_country_attribution__sample_counts.csv")
    if r3_models:
        save_table(pd.concat(r3_models, ignore_index=True), OUT_TABLES / "R3_model_paper_jsd_ols__country_attribution.csv")
    else:
        save_table(pd.DataFrame(), OUT_TABLES / "R3_model_paper_jsd_ols__country_attribution.csv")

    # =========================================================
    # 6. R4: alternative JSD outcomes
    # =========================================================

    r4_models = []
    for y_col in ["centroid_jsd", "max_pairwise_jsd"]:
        if y_col not in paper_main.columns:
            print(f"Skipping R4 {y_col}: column not found.")
            continue
        p_country = paper_main[(paper_main[STRICT_COUNTRY_COL] != "Unknown") & paper_main[y_col].notna()].copy()
        p_gender = paper_main[(paper_main[GENDER_CONF_COL].isin(["male", "female"])) & paper_main[y_col].notna()].copy()
        p_combined = paper_main[(paper_main[STRICT_COUNTRY_COL] != "Unknown") & paper_main[GENDER_CONF_COL].isin(["male", "female"]) & paper_main[y_col].notna()].copy()
        r4_models.append(fit_paper_jsd_ols(
            p_country,
            y_col=y_col,
            predictor_type="first_author_country",
            formula_rhs=f"C({STRICT_COUNTRY_COL}, Treatment(reference='{COUNTRY_REFERENCE}')) + {base_jsd_controls}",
            prefix=f"R4_{y_col}_country",
            country_col=STRICT_COUNTRY_COL,
        ))
        r4_models.append(fit_paper_jsd_ols(
            p_gender,
            y_col=y_col,
            predictor_type="first_author_gender",
            formula_rhs=f"C({GENDER_CONF_COL}, Treatment(reference='{GENDER_REFERENCE}')) + {base_jsd_controls}",
            prefix=f"R4_{y_col}_gender",
            gender_col=GENDER_CONF_COL,
        ))
        r4_models.append(fit_paper_jsd_ols(
            p_combined,
            y_col=y_col,
            predictor_type="country_and_gender",
            formula_rhs=f"C({STRICT_COUNTRY_COL}, Treatment(reference='{COUNTRY_REFERENCE}')) + C({GENDER_CONF_COL}, Treatment(reference='{GENDER_REFERENCE}')) + {base_jsd_controls}",
            prefix=f"R4_{y_col}_country_gender",
            country_col=STRICT_COUNTRY_COL,
            gender_col=GENDER_CONF_COL,
        ))

    r4_out = pd.concat([m for m in r4_models if m is not None and not m.empty], ignore_index=True) if r4_models else pd.DataFrame()
    save_table(r4_out, OUT_TABLES / "R4_model_paper_jsd_ols__alternative_jsd_outcomes.csv")

    # =========================================================
    # 7. R5: raw Genderize output sensitivity, if available
    # =========================================================

    raw_gender_col = first_existing_col(paper_main, [
        "first_author_name_inferred_gender",
        "first_author_gender_raw",
        "first_author_gender",
        "first_author_genderize_gender",
        "first_author_raw_gender",
    ])
    prob_col = first_existing_col(paper_main, [
        "first_author_gender_probability",
        "first_author_gender_prob",
        "first_author_genderize_probability",
        "first_author_raw_gender_probability",
    ])
    count_col = first_existing_col(paper_main, [
        "first_author_gender_count",
        "first_author_genderize_count",
        "first_author_raw_gender_count",
    ])

    if raw_gender_col is None:
        msg = pd.DataFrame([{
            "status": "skipped",
            "reason": "No raw first-author Genderize gender column found in paper_main/paper_background_features.",
            "checked_candidates": "first_author_name_inferred_gender; first_author_gender_raw; first_author_gender; first_author_genderize_gender; first_author_raw_gender",
        }])
        save_table(msg, OUT_TABLES / "R5_raw_genderize_sensitivity__SKIPPED.csv")
    else:
        p_raw = paper_main.copy()
        p_raw["first_author_gender_raw_api"] = p_raw[raw_gender_col].apply(clean_gender)
        if prob_col:
            p_raw["first_author_gender_raw_probability"] = pd.to_numeric(p_raw[prob_col], errors="coerce")
        if count_col:
            p_raw["first_author_gender_raw_count"] = pd.to_numeric(p_raw[count_col], errors="coerce")
        p_raw_gender = p_raw[p_raw["first_author_gender_raw_api"].isin(["male", "female"])].copy()
        p_raw_combined = p_raw[(p_raw[STRICT_COUNTRY_COL] != "Unknown") & p_raw["first_author_gender_raw_api"].isin(["male", "female"])].copy()
        r_raw_gender = align_reviews_to_papers(review_counts, p_raw_gender, extra_cols=["first_author_gender_raw_api"])
        r_raw_combined = align_reviews_to_papers(review_counts, p_raw_combined, extra_cols=["first_author_gender_raw_api"])

        save_table(p_raw_gender, OUT_SAMPLES_ROBUST / "R5_raw_genderize__paper_gender_known.parquet")
        save_table(r_raw_gender, OUT_SAMPLES_ROBUST / "R5_raw_genderize__review_gender_known.parquet")

        r5_counts = pd.DataFrame([
            sample_count(paper_main, "strict main paper sample"),
            sample_count(paper_main[paper_main[GENDER_CONF_COL].isin(["male", "female"])], "high-confidence gender paper sample"),
            sample_count(p_raw_gender, "raw Genderize male/female paper sample"),
            sample_count(r_raw_gender, "raw Genderize male/female review sample"),
            sample_count(p_raw_combined, "raw Genderize + country-known paper sample"),
            sample_count(r_raw_combined, "raw Genderize + country-known review sample"),
        ])
        save_table(r5_counts, OUT_TABLES / "R5_raw_genderize_sensitivity__sample_counts.csv")

        # Cross-tab high-confidence vs raw API gender.
        xtab = pd.crosstab(p_raw[GENDER_CONF_COL], p_raw["first_author_gender_raw_api"], dropna=False)
        xtab.to_csv(OUT_TABLES / "R5_raw_genderize_sensitivity__confident_vs_raw_crosstab.csv", encoding="utf-8-sig")

        r5_gender = fit_paper_jsd_ols(
            p_raw_gender,
            y_col=JSD_COL,
            predictor_type="first_author_gender_raw_api",
            formula_rhs=f"C(first_author_gender_raw_api, Treatment(reference='{GENDER_REFERENCE}')) + {base_jsd_controls}",
            prefix="R5_raw_genderize_gender",
            gender_col="first_author_gender_raw_api",
        )
        r5_combined = fit_paper_jsd_ols(
            p_raw_combined,
            y_col=JSD_COL,
            predictor_type="country_and_raw_gender",
            formula_rhs=f"C({STRICT_COUNTRY_COL}, Treatment(reference='{COUNTRY_REFERENCE}')) + C(first_author_gender_raw_api, Treatment(reference='{GENDER_REFERENCE}')) + {base_jsd_controls}",
            prefix="R5_raw_genderize_country_gender",
            country_col=STRICT_COUNTRY_COL,
            gender_col="first_author_gender_raw_api",
        )
        save_table(pd.concat([r5_gender, r5_combined], ignore_index=True), OUT_TABLES / "R5_model_paper_jsd_ols__raw_genderize.csv")
        chi_square_aspect_test(r_raw_gender, "first_author_gender_raw_api", "R5_raw_genderize_gender", exclude_groups=["Unknown"])


    # =========================================================
    # 8. R7: OLS ratio vs Poisson offset review-level models
    # =========================================================

    # Use strict samples; do not include international collaboration or n_author_countries.
    review_country = review_main[review_main[STRICT_COUNTRY_COL] != "Unknown"].copy()
    review_gender = review_main[review_main[GENDER_CONF_COL].isin(["male", "female"])].copy()
    review_combined = review_main[(review_main[STRICT_COUNTRY_COL] != "Unknown") & review_main[GENDER_CONF_COL].isin(["male", "female"])].copy()

    r7_counts = pd.DataFrame([
        sample_count(review_country, "R7 country review sample"),
        sample_count(review_gender, "R7 gender review sample"),
        sample_count(review_combined, "R7 country+gender review sample"),
    ])
    save_table(r7_counts, OUT_TABLES / "R7_review_model_ols_vs_poisson__sample_counts.csv")

    base_review_controls = "publication_year + C(subject_primary) + log_review_word_count + log_n_authors"

    r7_country_ols = fit_review_aspect_ols(
        review_country,
        predictor_type="first_author_country",
        formula_rhs=f"C({STRICT_COUNTRY_COL}, Treatment(reference='{COUNTRY_REFERENCE}')) + {base_review_controls}",
        prefix="R7_country",
        country_col=STRICT_COUNTRY_COL,
    )
    r7_country_pois = fit_review_aspect_poisson_offset(
        review_country,
        predictor_type="first_author_country",
        formula_rhs=f"C({STRICT_COUNTRY_COL}, Treatment(reference='{COUNTRY_REFERENCE}')) + {base_review_controls}",
        prefix="R7_country",
        country_col=STRICT_COUNTRY_COL,
    )
    save_table(r7_country_ols, OUT_TABLES / "R7_model_review_aspect_ratio_ols__country.csv")
    save_table(r7_country_pois, OUT_TABLES / "R7_model_review_aspect_count_poisson_offset__country.csv")
    compare_ols_poisson(r7_country_ols, r7_country_pois, "country")

    r7_gender_ols = fit_review_aspect_ols(
        review_gender,
        predictor_type="first_author_gender",
        formula_rhs=f"C({GENDER_CONF_COL}, Treatment(reference='{GENDER_REFERENCE}')) + {base_review_controls}",
        prefix="R7_gender",
        gender_col=GENDER_CONF_COL,
    )
    r7_gender_pois = fit_review_aspect_poisson_offset(
        review_gender,
        predictor_type="first_author_gender",
        formula_rhs=f"C({GENDER_CONF_COL}, Treatment(reference='{GENDER_REFERENCE}')) + {base_review_controls}",
        prefix="R7_gender",
        gender_col=GENDER_CONF_COL,
    )
    save_table(r7_gender_ols, OUT_TABLES / "R7_model_review_aspect_ratio_ols__gender.csv")
    save_table(r7_gender_pois, OUT_TABLES / "R7_model_review_aspect_count_poisson_offset__gender.csv")
    compare_ols_poisson(r7_gender_ols, r7_gender_pois, "gender")

    r7_combined_ols = fit_review_aspect_ols(
        review_combined,
        predictor_type="country_and_gender",
        formula_rhs=f"C({STRICT_COUNTRY_COL}, Treatment(reference='{COUNTRY_REFERENCE}')) + C({GENDER_CONF_COL}, Treatment(reference='{GENDER_REFERENCE}')) + {base_review_controls}",
        prefix="R7_country_gender",
        country_col=STRICT_COUNTRY_COL,
        gender_col=GENDER_CONF_COL,
    )
    r7_combined_pois = fit_review_aspect_poisson_offset(
        review_combined,
        predictor_type="country_and_gender",
        formula_rhs=f"C({STRICT_COUNTRY_COL}, Treatment(reference='{COUNTRY_REFERENCE}')) + C({GENDER_CONF_COL}, Treatment(reference='{GENDER_REFERENCE}')) + {base_review_controls}",
        prefix="R7_country_gender",
        country_col=STRICT_COUNTRY_COL,
        gender_col=GENDER_CONF_COL,
    )
    save_table(r7_combined_ols, OUT_TABLES / "R7_model_review_aspect_ratio_ols__country_gender.csv")
    save_table(r7_combined_pois, OUT_TABLES / "R7_model_review_aspect_count_poisson_offset__country_gender.csv")
    compare_ols_poisson(r7_combined_ols, r7_combined_pois, "country_gender")

    # =========================================================
    # 9. Final combined sample summary
    # =========================================================

    all_count_files = [
        OUT_TABLES / "R2_sample_robustness__sample_counts.csv",
        OUT_TABLES / "R3_country_attribution__sample_counts.csv",
        OUT_TABLES / "R5_raw_genderize_sensitivity__sample_counts.csv",
        OUT_TABLES / "R7_review_model_ols_vs_poisson__sample_counts.csv",
    ]
    count_tables = []
    for p in all_count_files:
        if p.exists():
            temp = pd.read_csv(p)
            temp.insert(0, "source_file", p.name)
            count_tables.append(temp)
    if count_tables:
        save_table(pd.concat(count_tables, ignore_index=True), OUT_TABLES / "ALL_first_batch_sample_counts.csv")

    print("\nDone. First-batch robustness outputs are in:")
    print(OUT_TABLES)


In [ ]:
run_first_batch()


## Second robustness batch

In [ ]:
def run_second_batch():
    # =========================================================
    # 0. Paths and constants
    # =========================================================

    OUT_MASTER = WORK_DIR
    OUT_MODEL = WORK_DIR
    OUT_SAMPLES = STRICT_SAMPLES
    OUT_SECOND = project_path("results/demo/robustness/second_batch")
    OUT_TABLES = OUT_SECOND / "tables"
    OUT_SAMPLES_SECOND = OUT_SECOND / "samples"

    for d in [OUT_TABLES, OUT_SAMPLES_SECOND]:
        d.mkdir(parents=True, exist_ok=True)

    ASPECT_CLASSES = [
        "Methodology",
        "Results",
        "Experiments and Evaluation",
        "Data and Datasets",
        "Figures and Tables",
        "Writing, Language and Terminology",
        "Theory and Models",
        "Comparison and Context",
        "Paper Assessment",
        "Novelty and Impact",
        "Validity, Rigor and Quality",
        "Limitations",
    ]

    JSD_COL = "mean_pairwise_jsd"
    COUNTRY_REFERENCE = ANALYSIS["country_reference"]
    PAIR_REFERENCE = "MM"
    COUNTRY_MIN_PAPERS = int(CONFIG["demo"]["country_min_papers"])  # DEMO ONLY; formal value remains 500
    STRICT_COUNTRY_COL = "first_author_country_group_n500_strict"
    GENDER_CONF_COL = "first_author_gender_confident"

    # =========================================================
    # 1. General utilities
    # =========================================================

    def save_table(df: pd.DataFrame, path: Path) -> None:
        path.parent.mkdir(parents=True, exist_ok=True)
        if path.suffix == ".parquet":
            df.to_parquet(path, index=False)
        elif path.suffix == ".csv":
            df.to_csv(path, index=False, encoding="utf-8-sig")
        else:
            raise ValueError(f"Unsupported output format: {path}")
        print(f"Saved: {path} | rows={len(df):,}, cols={df.shape[1]}")


    def read_required(path: Path) -> pd.DataFrame:
        if not path.exists():
            raise FileNotFoundError(f"Required file not found: {path}")
        print(f"Reading: {path}")
        return pd.read_parquet(path)


    def read_optional(paths: Sequence[Path]) -> pd.DataFrame | None:
        for p in paths:
            if p.exists():
                print(f"Reading optional: {p}")
                return pd.read_parquet(p)
        return None


    def clean_value(x, unknown: str = "Unknown") -> str:
        if pd.isna(x):
            return unknown
        s = str(x).strip()
        if s == "" or s.lower() in {"nan", "none", "null", "unknown", "na", "n/a"}:
            return unknown
        return s


    def normalize_gender(x) -> str:
        s = clean_value(x).lower()
        if s in {"male", "m"}:
            return "male"
        if s in {"female", "f"}:
            return "female"
        return "Unknown"


    def safe_log1p(s: pd.Series) -> pd.Series:
        return np.log1p(pd.to_numeric(s, errors="coerce").fillna(0).clip(lower=0))


    def first_token_country(x) -> str:
        """Return a reproducible country code from a possibly multi-country string."""
        s = clean_value(x)
        if s == "Unknown":
            return "Unknown"
        parts = re.split(r"\s*\|\s*|\s*;\s*|\s*,\s*", s)
        parts = [p.strip().upper() for p in parts if p.strip()]
        if not parts:
            return "Unknown"
        return parts[0]


    def make_country_group(df: pd.DataFrame, raw_col: str, out_col: str, min_papers: int = COUNTRY_MIN_PAPERS) -> pd.Series:
        raw = df[raw_col].apply(first_token_country)
        counts = raw[raw != "Unknown"].value_counts()
        keep = set(counts[counts >= min_papers].index)
        return raw.apply(lambda x: x if x in keep else ("Unknown" if x == "Unknown" else "Other"))


    def ensure_country_group(df: pd.DataFrame, raw_candidates: list[str], group_candidates: list[str], out_col: str) -> tuple[pd.DataFrame, str | None]:
        """Create a strict-sample n>=500 country group when possible.

        Preference order:
        1) raw country column, then rebuild n>=500 groups on the current strict sample.
        2) existing grouped column, if raw column is not available.
        """
        out = df.copy()
        raw_col = next((c for c in raw_candidates if c in out.columns), None)
        if raw_col is not None:
            out[out_col] = make_country_group(out, raw_col, out_col)
            return out, out_col

        group_col = next((c for c in group_candidates if c in out.columns), None)
        if group_col is not None:
            out[out_col] = out[group_col].apply(clean_value)
            return out, out_col

        return out, None


    def best_review_count_sum(df: pd.DataFrame) -> int:
        for c in ["n_reviews", "n_reviews_total", "n_reviews_valid_for_jsd", "n_reviews_valid"]:
            if c in df.columns:
                return int(pd.to_numeric(df[c], errors="coerce").fillna(0).sum())
        return int(len(df))


    def sample_row(name: str, paper_df: pd.DataFrame, review_df: pd.DataFrame | None = None) -> dict:
        if review_df is not None:
            n_reviews = len(review_df)
        else:
            n_reviews = best_review_count_sum(paper_df)
        return {"sample": name, "n_papers": paper_df["paper_code"].nunique(), "n_reviews": int(n_reviews)}


    def coerce_for_statsmodels(df: pd.DataFrame, numeric_cols: list[str], categorical_cols: list[str]) -> pd.DataFrame:
        out = df.copy()
        for col in numeric_cols:
            if col in out.columns:
                out[col] = pd.to_numeric(out[col], errors="coerce").astype("float64")
        for col in categorical_cols:
            if col in out.columns:
                out[col] = out[col].astype("object").where(out[col].notna(), "Unknown").astype(str)
        if "paper_code" in out.columns:
            out["paper_code"] = out["paper_code"].astype("object").astype(str)
        for col in out.columns:
            if pd.api.types.is_extension_array_dtype(out[col].dtype):
                if pd.api.types.is_numeric_dtype(out[col].dtype):
                    out[col] = pd.to_numeric(out[col], errors="coerce").astype("float64")
                else:
                    out[col] = out[col].astype("object").where(out[col].notna(), "Unknown").astype(str)
        return out


    def cramers_v_from_table(tab: pd.DataFrame) -> tuple[float, float, int, float]:
        chi2, p, dof, _ = chi2_contingency(tab.to_numpy(dtype=float))  # Numeric adapter for nullable pandas dtypes
        n = tab.to_numpy().sum()
        r, k = tab.shape
        v = np.sqrt((chi2 / n) / max(1, min(r - 1, k - 1))) if n > 0 else np.nan
        return chi2, p, dof, v


    def chi_square_aspects(review_df: pd.DataFrame, group_col: str, prefix: str, exclude_unknown: bool = True) -> pd.DataFrame:
        temp = review_df.copy()
        temp[group_col] = temp[group_col].apply(clean_value)
        if exclude_unknown:
            temp = temp[temp[group_col] != "Unknown"].copy()

        aspect_cols = [c for c in ASPECT_CLASSES if c in temp.columns]
        tab = temp.groupby(group_col)[aspect_cols].sum()
        tab = tab.loc[tab.sum(axis=1) > 0, tab.sum(axis=0) > 0]

        if tab.shape[0] < 2 or tab.shape[1] < 2:
            out = pd.DataFrame([{
                "group_col": group_col,
                "n_groups": tab.shape[0],
                "n_aspects": tab.shape[1],
                "n_aspect_mentions": float(tab.to_numpy().sum()) if tab.size else 0,
                "chi2": np.nan,
                "dof": np.nan,
                "p_value": np.nan,
                "cramers_v": np.nan,
            }])
        else:
            chi2, p, dof, v = cramers_v_from_table(tab)
            out = pd.DataFrame([{
                "group_col": group_col,
                "n_groups": tab.shape[0],
                "n_aspects": tab.shape[1],
                "n_aspect_mentions": float(tab.to_numpy().sum()),
                "chi2": chi2,
                "dof": dof,
                "p_value": p,
                "cramers_v": v,
            }])

        save_table(tab.reset_index(), OUT_TABLES / f"{prefix}__aspect_contingency.csv")
        save_table(out, OUT_TABLES / f"{prefix}__chi_square_aspect.csv")
        return out


    def chi_square_cross(paper_df: pd.DataFrame, row_col: str, col_col: str, prefix: str, exclude_unknown_rows: bool = True, exclude_unknown_cols: bool = False) -> pd.DataFrame:
        temp = paper_df.copy()
        temp[row_col] = temp[row_col].apply(clean_value)
        temp[col_col] = temp[col_col].apply(clean_value)
        if exclude_unknown_rows:
            temp = temp[temp[row_col] != "Unknown"]
        if exclude_unknown_cols:
            temp = temp[temp[col_col] != "Unknown"]

        tab = pd.crosstab(temp[row_col], temp[col_col])
        tab = tab.loc[tab.sum(axis=1) > 0, tab.sum(axis=0) > 0]

        if tab.shape[0] < 2 or tab.shape[1] < 2:
            stats = pd.DataFrame([{
                "row_col": row_col,
                "col_col": col_col,
                "n_rows": tab.shape[0],
                "n_cols": tab.shape[1],
                "n_papers": int(tab.to_numpy().sum()) if tab.size else 0,
                "chi2": np.nan,
                "dof": np.nan,
                "p_value": np.nan,
                "cramers_v": np.nan,
            }])
        else:
            chi2, p, dof, v = cramers_v_from_table(tab)
            stats = pd.DataFrame([{
                "row_col": row_col,
                "col_col": col_col,
                "n_rows": tab.shape[0],
                "n_cols": tab.shape[1],
                "n_papers": int(tab.to_numpy().sum()),
                "chi2": chi2,
                "dof": dof,
                "p_value": p,
                "cramers_v": v,
            }])

        row_pct = tab.div(tab.sum(axis=1), axis=0).fillna(0)
        save_table(tab.reset_index(), OUT_TABLES / f"{prefix}__counts.csv")
        save_table(row_pct.reset_index(), OUT_TABLES / f"{prefix}__row_share.csv")
        save_table(stats, OUT_TABLES / f"{prefix}__chi_square.csv")
        return stats


    def descriptive_jsd_by_group(df: pd.DataFrame, group_col: str, prefix: str, exclude_unknown: bool = True) -> pd.DataFrame:
        temp = df.copy()
        temp[group_col] = temp[group_col].apply(clean_value)
        if exclude_unknown:
            temp = temp[temp[group_col] != "Unknown"]

        rows = []
        for g, sub in temp.groupby(group_col):
            vals = pd.to_numeric(sub[JSD_COL], errors="coerce").dropna()
            rows.append({
                "group_col": group_col,
                "group": g,
                "n_papers": int(sub["paper_code"].nunique()),
                "n_reviews": best_review_count_sum(sub),
                "mean_jsd": vals.mean(),
                "median_jsd": vals.median(),
                "sd_jsd": vals.std(),
                "q25_jsd": vals.quantile(0.25),
                "q75_jsd": vals.quantile(0.75),
            })
        out = pd.DataFrame(rows).sort_values("n_papers", ascending=False)
        save_table(out, OUT_TABLES / f"{prefix}__descriptive_jsd.csv")
        return out


    def fit_jsd_model(df: pd.DataFrame, formula_rhs: str, prefix: str, group_cols: list[str]) -> pd.DataFrame:
        controls = [
            JSD_COL,
            "paper_code",
            "publication_year",
            "subject_primary",
            "n_reviews_valid_for_jsd",
            "log_aspect_total_count",
            "log_n_authors",
        ] + group_cols
        cols = [c for c in controls if c in df.columns]
        model_df = df[cols].dropna().copy()
        if model_df.empty:
            out = pd.DataFrame()
            save_table(out, OUT_TABLES / f"{prefix}.csv")
            return out

        model_df = coerce_for_statsmodels(
            model_df,
            numeric_cols=[JSD_COL, "publication_year", "n_reviews_valid_for_jsd", "log_aspect_total_count", "log_n_authors"],
            categorical_cols=["subject_primary"] + group_cols,
        )

        formula = f"{JSD_COL} ~ {formula_rhs}"
        model = smf.ols(formula, data=model_df).fit(cov_type="HC3")
        rows = []
        for term, est in model.params.items():
            if term == "Intercept":
                continue
            conf = model.conf_int().loc[term]
            rows.append({
                "term": term,
                "estimate": est,
                "conf_low": conf[0],
                "conf_high": conf[1],
                "p_value": model.pvalues.get(term, np.nan),
                "n_papers": int(model.nobs),
                "model": prefix,
            })
        out = pd.DataFrame(rows)
        if len(out) > 0:
            out["p_adj_bh"] = multipletests(out["p_value"].fillna(1), method="fdr_bh")[1]
        save_table(out, OUT_TABLES / f"{prefix}.csv")
        return out


    def grouped_numeric_summary(df: pd.DataFrame, group_col: str, value_cols: list[str], prefix: str) -> pd.DataFrame:
        rows = []
        for g, sub in df.groupby(group_col):
            row_base = {"group_col": group_col, "group": g, "n_papers": sub["paper_code"].nunique(), "n_reviews": best_review_count_sum(sub)}
            for c in value_cols:
                if c not in sub.columns:
                    continue
                vals = pd.to_numeric(sub[c], errors="coerce").dropna()
                row_base[f"{c}__mean"] = vals.mean()
                row_base[f"{c}__median"] = vals.median()
                row_base[f"{c}__sd"] = vals.std()
                row_base[f"{c}__q25"] = vals.quantile(0.25)
                row_base[f"{c}__q75"] = vals.quantile(0.75)
            rows.append(row_base)
        out = pd.DataFrame(rows)
        save_table(out, OUT_TABLES / f"{prefix}.csv")
        return out


    def mannwhitney_known_unknown(df: pd.DataFrame, status_col: str, value_cols: list[str], prefix: str) -> pd.DataFrame:
        rows = []
        for c in value_cols:
            if c not in df.columns:
                continue
            a = pd.to_numeric(df.loc[df[status_col] == "Known", c], errors="coerce").dropna()
            b = pd.to_numeric(df.loc[df[status_col] == "Unknown", c], errors="coerce").dropna()
            if len(a) == 0 or len(b) == 0:
                stat, p = np.nan, np.nan
            else:
                stat, p = mannwhitneyu(a, b, alternative="two-sided")
            rows.append({
                "variable": c,
                "known_n": len(a),
                "unknown_n": len(b),
                "known_mean": a.mean(),
                "unknown_mean": b.mean(),
                "known_median": a.median(),
                "unknown_median": b.median(),
                "mannwhitney_u": stat,
                "p_value": p,
            })
        out = pd.DataFrame(rows)
        if len(out) > 0:
            out["p_adj_bh"] = multipletests(out["p_value"].fillna(1), method="fdr_bh")[1]
        save_table(out, OUT_TABLES / f"{prefix}.csv")
        return out

    # =========================================================
    # 2. Load strict main samples and enrich if needed
    # =========================================================

    paper_main = read_required(OUT_SAMPLES / "main_strict_paper_jsd_no_large50.parquet")
    review_main = read_required(OUT_SAMPLES / "main_strict_review_aligned.parquet")

    # Optional background table, used only to recover variables missing from the saved analysis samples.
    bg = read_optional([
        OUT_MASTER / "paper_background_features_v2.parquet",
        OUT_MODEL / "paper_background_features_v2.parquet",
    ])

    if bg is not None and "paper_code" in bg.columns:
        missing_cols = [
            c for c in bg.columns
            if c not in paper_main.columns and c not in {"paper_code"}
        ]
        if missing_cols:
            paper_main = paper_main.merge(bg[["paper_code"] + missing_cols], on="paper_code", how="left")

    # Ensure basic controls.
    if "log_n_authors" not in paper_main.columns and "n_authors" in paper_main.columns:
        paper_main["log_n_authors"] = safe_log1p(paper_main["n_authors"])
    if "log_aspect_total_count" not in paper_main.columns and "aspect_total_count" in paper_main.columns:
        paper_main["log_aspect_total_count"] = safe_log1p(paper_main["aspect_total_count"])

    # Merge paper-level variables back to review_main if missing.
    merge_cols = [c for c in paper_main.columns if c not in review_main.columns and c != "paper_code"]
    if merge_cols:
        review_main = review_main.merge(paper_main[["paper_code"] + merge_cols], on="paper_code", how="left")

    # Normalize key variables.
    paper_main[GENDER_CONF_COL] = paper_main.get(GENDER_CONF_COL, pd.Series("Unknown", index=paper_main.index)).apply(normalize_gender)
    if STRICT_COUNTRY_COL not in paper_main.columns:
        first_country_raw = next((c for c in ["first_author_country", "first_author_country_primary", "first_author_affil_country", "first_author_country_code"] if c in paper_main.columns), None)
        if first_country_raw is not None:
            paper_main[STRICT_COUNTRY_COL] = make_country_group(paper_main, first_country_raw, STRICT_COUNTRY_COL)
        else:
            paper_main[STRICT_COUNTRY_COL] = "Unknown"

    review_main[GENDER_CONF_COL] = review_main.get(GENDER_CONF_COL, pd.Series("Unknown", index=review_main.index)).apply(normalize_gender)
    if STRICT_COUNTRY_COL not in review_main.columns:
        review_main = review_main.merge(paper_main[["paper_code", STRICT_COUNTRY_COL]], on="paper_code", how="left")

    # Keep only review records aligned to the strict main paper sample.
    review_main = review_main[review_main["paper_code"].isin(set(paper_main["paper_code"]))].copy()

    # Save base count.
    sample_counts = [sample_row("strict_main_base", paper_main, review_main)]


    # =========================================================
    # R3b. Alternative country attribution for JSD models
    # =========================================================

    base_controls = "publication_year + C(subject_primary) + n_reviews_valid_for_jsd + log_aspect_total_count + log_n_authors"

    alt_country_specs = {
        "corresponding_author_country": {
            "raw_candidates": [
                "corresponding_author_country",
                "corresponding_author_country_primary",
                "corr_author_country",
                "corr_authors_country",
                "corr_author_country_primary",
            ],
            "group_candidates": [
                "corresponding_author_country_group_n500_strict",
                "corresponding_author_country_group_n500",
            ],
            "out_col": "corresponding_author_country_group_n500_strict_rebuilt",
        },
        "dominant_country": {
            "raw_candidates": ["dominant_country", "paper_dominant_country", "dominant_author_country"],
            "group_candidates": ["dominant_country_group_n500_strict", "dominant_country_group_n500"],
            "out_col": "dominant_country_group_n500_strict_rebuilt",
        },
    }

    for name, spec in alt_country_specs.items():
        temp, col = ensure_country_group(
            paper_main,
            raw_candidates=spec["raw_candidates"],
            group_candidates=spec["group_candidates"],
            out_col=spec["out_col"],
        )
        if col is None:
            save_table(pd.DataFrame([{"analysis": f"R3b_{name}", "status": "SKIPPED", "reason": "No usable country column found"}]), OUT_TABLES / f"R3b_model_paper_jsd_ols__{name}__SKIPPED.csv")
            continue

        sub = temp[temp[col] != "Unknown"].copy()
        sample_counts.append(sample_row(f"R3b_{name}_known", sub))
        save_table(sub[["paper_code", col]], OUT_SAMPLES_SECOND / f"R3b_sample__{name}.parquet")

        # Descriptive JSD and OLS model.
        descriptive_jsd_by_group(sub, col, f"R3b_{name}", exclude_unknown=False)
        if COUNTRY_REFERENCE in set(sub[col]):
            rhs = f"C({col}, Treatment(reference='{COUNTRY_REFERENCE}')) + {base_controls}"
        else:
            rhs = f"C({col}) + {base_controls}"
        fit_jsd_model(sub, rhs, f"R3b_model_paper_jsd_ols__{name}", group_cols=[col])

        # Review-level chi-square aspect test with the same alternative country group.
        rsub = review_main.merge(sub[["paper_code", col]], on="paper_code", how="inner")
        sample_counts.append(sample_row(f"R3b_{name}_review_known", sub, rsub))
        chi_square_aspects(rsub, col, f"R3b_{name}", exclude_unknown=False)


    # =========================================================
    # R6. Gender Unknown / missingness-bias checks
    # =========================================================

    paper_r6 = paper_main.copy()
    paper_r6["gender_known_status"] = np.where(paper_r6[GENDER_CONF_COL].isin(["male", "female"]), "Known", "Unknown")
    review_r6 = review_main.merge(paper_r6[["paper_code", "gender_known_status"]], on="paper_code", how="left", suffixes=("", "_paper_r6"))

    sample_counts.append(sample_row("R6_gender_known_status_base", paper_r6, review_r6))

    # Overall counts.
    r6_overall = paper_r6.groupby("gender_known_status", dropna=False).agg(
        n_papers=("paper_code", "nunique"),
    ).reset_index()
    review_counts = review_r6.groupby("gender_known_status", dropna=False).size().rename("n_reviews").reset_index()
    r6_overall = r6_overall.merge(review_counts, on="gender_known_status", how="left")
    r6_overall["paper_share"] = r6_overall["n_papers"] / r6_overall["n_papers"].sum()
    r6_overall["review_share"] = r6_overall["n_reviews"] / r6_overall["n_reviews"].sum()
    save_table(r6_overall, OUT_TABLES / "R6_gender_known_unknown__overall_counts.csv")

    # Known/Unknown by country, subject, and year.
    for col, prefix in [
        (STRICT_COUNTRY_COL, "R6_gender_known_unknown_by_country"),
        ("subject_primary", "R6_gender_known_unknown_by_subject"),
        ("publication_year", "R6_gender_known_unknown_by_year"),
    ]:
        if col not in paper_r6.columns:
            continue
        tab = pd.crosstab(paper_r6[col].apply(clean_value), paper_r6["gender_known_status"])
        if "Unknown" in tab.index and col == STRICT_COUNTRY_COL:
            # Keep country Unknown in the output table, but the chi-square test below can be interpreted cautiously.
            pass
        tab["Total"] = tab.sum(axis=1)
        if "Unknown" not in tab.columns:
            tab["Unknown"] = 0
        if "Known" not in tab.columns:
            tab["Known"] = 0
        tab["unknown_share"] = tab["Unknown"] / tab["Total"].replace(0, np.nan)
        save_table(tab.reset_index().rename(columns={col: "group"}), OUT_TABLES / f"{prefix}.csv")
        chi_square_cross(paper_r6, col, "gender_known_status", prefix=f"{prefix}", exclude_unknown_rows=False, exclude_unknown_cols=False)

    # Known/Unknown differences in paper characteristics.
    value_cols = [
        JSD_COL,
        "n_authors",
        "n_reviews_valid_for_jsd",
        "aspect_total_count",
        "log_aspect_total_count",
    ]
    grouped_numeric_summary(paper_r6, "gender_known_status", value_cols, "R6_gender_known_unknown__paper_characteristics")
    mannwhitney_known_unknown(paper_r6, "gender_known_status", value_cols, "R6_gender_known_unknown__paper_characteristics_mannwhitney")

    # Optional: distinguish high-confidence known, low-confidence raw known, and raw unknown if raw gender exists.
    raw_gender_candidates = [
        "first_author_gender_raw",
        "first_author_name_inferred_gender",
        "first_author_gender",
        "first_author_genderize_gender",
        "first_author_raw_gender",
    ]
    raw_col = next((c for c in raw_gender_candidates if c in paper_r6.columns), None)
    if raw_col is not None:
        raw_gender = paper_r6[raw_col].apply(normalize_gender)
        paper_r6["gender_confidence_status_3way"] = np.select(
            [
                paper_r6[GENDER_CONF_COL].isin(["male", "female"]),
                (~paper_r6[GENDER_CONF_COL].isin(["male", "female"])) & raw_gender.isin(["male", "female"]),
                raw_gender.eq("Unknown"),
            ],
            ["High-confidence known", "Low-confidence raw known", "Raw unknown"],
            default="Other/unclear",
        )
        grouped_numeric_summary(paper_r6, "gender_confidence_status_3way", value_cols, "R6_gender_confidence_3way__paper_characteristics")
        save_table(
            paper_r6.groupby("gender_confidence_status_3way").agg(n_papers=("paper_code", "nunique")).reset_index(),
            OUT_TABLES / "R6_gender_confidence_3way__counts.csv",
        )
    else:
        save_table(pd.DataFrame([{"analysis": "R6_gender_confidence_3way", "status": "SKIPPED", "reason": "No raw Genderize gender column found"}]), OUT_TABLES / "R6_gender_confidence_3way__SKIPPED.csv")


    # =========================================================
    # R8. First-last author gender-pair supplementary analysis
    # =========================================================

    def normalize_pair(x) -> str:
        s = clean_value(x).upper().replace("-", "").replace("_", "").replace(" ", "")
        mapping = {
            "MM": "MM",
            "MW": "MW",
            "WM": "WM",
            "WW": "WW",
            "MALEMALE": "MM",
            "MALEFEMALE": "MW",
            "FEMALEMALE": "WM",
            "FEMALEFEMALE": "WW",
        }
        return mapping.get(s, "Unknown")

    pair_col = "first_last_gender_pair"
    if pair_col not in paper_main.columns:
        save_table(pd.DataFrame([{"analysis": "R8_first_last_gender_pair", "status": "SKIPPED", "reason": "first_last_gender_pair column not found"}]), OUT_TABLES / "R8_first_last_gender_pair__SKIPPED.csv")
    else:
        paper_r8 = paper_main.copy()
        paper_r8[pair_col] = paper_r8[pair_col].apply(normalize_pair)
        known_pairs = ["MM", "MW", "WM", "WW"]
        paper_r8_known = paper_r8[paper_r8[pair_col].isin(known_pairs)].copy()

        review_base_for_pair = review_main.drop(columns=[pair_col], errors="ignore").copy()

        review_r8_known = review_base_for_pair.merge(
            paper_r8_known[["paper_code", pair_col]],
            on="paper_code",
            how="inner",
        )

        sample_counts.append(sample_row("R8_first_last_gender_pair_known", paper_r8_known, review_r8_known))
        save_table(paper_r8_known, OUT_SAMPLES_SECOND / "R8_sample__first_last_gender_pair_known.parquet")

        # Counts and descriptive JSD.
        pair_counts = paper_r8.groupby(pair_col).agg(
            n_papers=("paper_code", "nunique")
        ).reset_index()

        review_pair_counts = review_base_for_pair.merge(
            paper_r8[["paper_code", pair_col]],
            on="paper_code",
            how="inner",
        ).groupby(pair_col).size().rename("n_reviews").reset_index()

        pair_counts = pair_counts.merge(review_pair_counts, on=pair_col, how="left")
        pair_counts["paper_share"] = pair_counts["n_papers"] / pair_counts["n_papers"].sum()
        save_table(pair_counts, OUT_TABLES / "R8_first_last_gender_pair__counts.csv")

        descriptive_jsd_by_group(paper_r8_known, pair_col, "R8_first_last_gender_pair", exclude_unknown=False)

        # Kruskal test for group differences in JSD.
        groups = [pd.to_numeric(g[JSD_COL], errors="coerce").dropna() for _, g in paper_r8_known.groupby(pair_col)]
        if len(groups) >= 2 and all(len(g) > 0 for g in groups):
            stat, p = kruskal(*groups)
        else:
            stat, p = np.nan, np.nan
        save_table(pd.DataFrame([{
            "group_col": pair_col,
            "n_groups": paper_r8_known[pair_col].nunique(),
            "n_papers": paper_r8_known["paper_code"].nunique(),
            "kruskal_h": stat,
            "p_value": p,
        }]), OUT_TABLES / "R8_first_last_gender_pair__kruskal_jsd.csv")

        # OLS JSD model.
        ref = PAIR_REFERENCE if PAIR_REFERENCE in set(paper_r8_known[pair_col]) else sorted(paper_r8_known[pair_col].unique())[0]
        rhs = f"C({pair_col}, Treatment(reference='{ref}')) + {base_controls}"
        fit_jsd_model(paper_r8_known, rhs, "R8_model_paper_jsd_ols__first_last_gender_pair", group_cols=[pair_col])

        # Review-level aspect distribution test.
        chi_square_aspects(review_r8_known, pair_col, "R8_first_last_gender_pair", exclude_unknown=False)


    # =========================================================
    # R9. Subject-composition checks by author background
    # =========================================================

    # Main subject-composition checks.
    if "subject_primary" in paper_main.columns:
        if STRICT_COUNTRY_COL in paper_main.columns:
            chi_square_cross(
                paper_main[paper_main[STRICT_COUNTRY_COL] != "Unknown"].copy(),
                STRICT_COUNTRY_COL,
                "subject_primary",
                prefix="R9_subject_composition_by_first_author_country",
                exclude_unknown_rows=False,
                exclude_unknown_cols=False,
            )

        gender_known = paper_main[paper_main[GENDER_CONF_COL].isin(["male", "female"])].copy()
        if len(gender_known) > 0:
            chi_square_cross(
                gender_known,
                GENDER_CONF_COL,
                "subject_primary",
                prefix="R9_subject_composition_by_first_author_gender",
                exclude_unknown_rows=False,
                exclude_unknown_cols=False,
            )

        # Optional: raw Genderize subject-composition sensitivity, if raw gender exists.
        if raw_col is not None:
            raw_gender = paper_main[raw_col].apply(normalize_gender)
            raw_gender_df = paper_main.copy()
            raw_gender_df["first_author_gender_raw_api"] = raw_gender
            raw_gender_known = raw_gender_df[raw_gender_df["first_author_gender_raw_api"].isin(["male", "female"])].copy()
            if len(raw_gender_known) > 0:
                chi_square_cross(
                    raw_gender_known,
                    "first_author_gender_raw_api",
                    "subject_primary",
                    prefix="R9_subject_composition_by_raw_genderize_gender",
                    exclude_unknown_rows=False,
                    exclude_unknown_cols=False,
                )

    # Save final sample counts.
    save_table(pd.DataFrame(sample_counts), OUT_TABLES / "ALL_second_batch_sample_counts.csv")

    print("\nDone. Outputs written to:")
    print(f"  {OUT_TABLES}")
    print(f"  {OUT_SAMPLES_SECOND}")


In [ ]:
run_second_batch()


## Eight high-coverage aspects

In [ ]:
def run_high_coverage_8():
    # =========================================================
    # 0. Paths and constants
    # =========================================================

    OUT_SAMPLES = project_path("results/demo/robustness/high_coverage/samples")
    OUT_TABLES = project_path("results/demo/robustness/high_coverage/tables")
    OUT_FIG = project_path("results/demo/robustness/high_coverage/figures")

    for d in [OUT_TABLES, OUT_FIG, OUT_SAMPLES]:
        d.mkdir(parents=True, exist_ok=True)

    ALL_ASPECT_CLASSES = [
        "Methodology",
        "Results",
        "Experiments and Evaluation",
        "Data and Datasets",
        "Figures and Tables",
        "Writing, Language and Terminology",
        "Theory and Models",
        "Comparison and Context",
        "Paper Assessment",
        "Novelty and Impact",
        "Validity, Rigor and Quality",
        "Limitations",
    ]

    LOW_COVERAGE_ASPECTS = [
        "Theory and Models",
        "Limitations",
        "Validity, Rigor and Quality",
        "Paper Assessment",
    ]

    # Follow-up analysis: 8 high-coverage aspects.
    ASPECT_CLASSES = [c for c in ALL_ASPECT_CLASSES if c not in LOW_COVERAGE_ASPECTS]

    ASPECT_SHORT = {
        "Methodology": "Methodology",
        "Results": "Results",
        "Experiments and Evaluation": "Experiments & evaluation",
        "Data and Datasets": "Data & datasets",
        "Figures and Tables": "Figures & tables",
        "Writing, Language and Terminology": "Writing & language",
        "Theory and Models": "Theory & models",
        "Comparison and Context": "Comparison & context",
        "Paper Assessment": "Paper assessment",
        "Novelty and Impact": "Novelty & impact",
        "Validity, Rigor and Quality": "Validity & rigor",
        "Limitations": "Limitations",
    }

    COUNTRY_COL = "first_author_country_group_n500_strict"
    GENDER_COL = "first_author_gender_confident"
    COUNTRY_REFERENCE = ANALYSIS["country_reference"]
    GENDER_REFERENCE = ANALYSIS["gender_reference"]
    JSD_COL_8 = "mean_pairwise_jsd_8aspect"
    CENTROID_JSD_COL_8 = "centroid_jsd_8aspect"
    MAX_JSD_COL_8 = "max_pairwise_jsd_8aspect"
    ASPECT_TOTAL_8 = "aspect_total_count_8aspect"
    N_REVIEWS_VALID_8 = "n_reviews_valid_for_jsd_8aspect"

    DIVERGING_CMAP = LinearSegmentedColormap.from_list(
        "blue_white_orange",
        ["#4C78A8", "#F7F7F7", "#F58518"],
        N=256,
    )

    # =========================================================
    # 1. Utilities
    # =========================================================

    def save_table(df: pd.DataFrame, path: Path) -> None:
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        if path.suffix == ".parquet":
            df.to_parquet(path, index=False)
        elif path.suffix == ".csv":
            df.to_csv(path, index=False, encoding="utf-8-sig")
        else:
            raise ValueError(f"Unsupported output format: {path}")
        print(f"Saved: {path} | rows={len(df):,}, cols={df.shape[1]}")


    def save_fig(name: str) -> None:
        path = OUT_FIG / name
        plt.tight_layout()
        plt.savefig(path, dpi=300, bbox_inches="tight")
        print(f"Saved: {path}")


    def read_required(path: Path) -> pd.DataFrame:
        if not path.exists():
            raise FileNotFoundError(
                f"Required file not found: {path}\n"
                "Please run the 12-aspect strict main analysis first."
            )
        print(f"Reading: {path}")
        return pd.read_parquet(path)


    def clean_value(x, unknown: str = "Unknown") -> str:
        if pd.isna(x):
            return unknown
        x = str(x).strip()
        if x == "" or x.lower() in {"nan", "none", "null", "na"}:
            return unknown
        return x


    def clean_gender(x) -> str:
        x = clean_value(x).lower()
        if x in {"male", "m"}:
            return "male"
        if x in {"female", "f", "woman", "women"}:
            return "female"
        return "Unknown"


    def safe_log1p(series: pd.Series) -> pd.Series:
        s = pd.to_numeric(series, errors="coerce")
        s = s.where(s >= 0)
        return np.log1p(s)


    def aspect_ratio_col(cls: str) -> str:
        return f"ratio8__{re.sub(r'[^A-Za-z0-9]+', '_', cls).strip('_')}"


    def share_col(cls: str) -> str:
        return f"share8__{re.sub(r'[^A-Za-z0-9]+', '_', cls).strip('_')}"


    def prepare_categorical_and_numeric(df: pd.DataFrame) -> pd.DataFrame:
        out = df.copy()

        if COUNTRY_COL in out.columns:
            out[COUNTRY_COL] = out[COUNTRY_COL].apply(clean_value)
        else:
            raise KeyError(f"Missing country grouping column: {COUNTRY_COL}")

        if GENDER_COL in out.columns:
            out[GENDER_COL] = out[GENDER_COL].apply(clean_gender)
        else:
            out[GENDER_COL] = "Unknown"

        if "subject_primary" in out.columns:
            out["subject_primary"] = out["subject_primary"].apply(clean_value)
        else:
            out["subject_primary"] = "Unknown"

        for col in [
            "publication_year",
            "n_authors",
            "n_reviews",
            "n_reviews_valid_for_jsd",
            "aspect_total_count",
            ASPECT_TOTAL_8,
            N_REVIEWS_VALID_8,
            JSD_COL_8,
            CENTROID_JSD_COL_8,
            MAX_JSD_COL_8,
        ]:
            if col in out.columns:
                out[col] = pd.to_numeric(out[col], errors="coerce")

        if "log_n_authors" not in out.columns:
            out["log_n_authors"] = safe_log1p(out["n_authors"])
        else:
            out["log_n_authors"] = pd.to_numeric(out["log_n_authors"], errors="coerce")

        out["log_aspect_total_count_8aspect"] = safe_log1p(out[ASPECT_TOTAL_8]) if ASPECT_TOTAL_8 in out.columns else np.nan
        return out


    def coerce_for_statsmodels(
        df: pd.DataFrame,
        numeric_cols: Sequence[str] | None = None,
        categorical_cols: Sequence[str] | None = None,
        group_col: str | None = None,
    ) -> pd.DataFrame:
        """Convert pandas nullable dtypes to statsmodels-friendly dtypes."""
        out = df.copy()
        numeric_cols = list(numeric_cols or [])
        categorical_cols = list(categorical_cols or [])

        for col in numeric_cols:
            if col in out.columns:
                out[col] = pd.to_numeric(out[col], errors="coerce").astype("float64")

        for col in categorical_cols:
            if col in out.columns:
                out[col] = out[col].astype("object").where(out[col].notna(), "Unknown").astype(str)

        if group_col is not None and group_col in out.columns:
            out[group_col] = out[group_col].astype("object").astype(str)

        for col in out.columns:
            if pd.api.types.is_extension_array_dtype(out[col].dtype):
                if pd.api.types.is_numeric_dtype(out[col].dtype):
                    out[col] = pd.to_numeric(out[col], errors="coerce").astype("float64")
                elif pd.api.types.is_bool_dtype(out[col].dtype):
                    out[col] = out[col].astype("float64")
                else:
                    out[col] = out[col].astype("object").where(out[col].notna(), "Unknown").astype(str)
        return out


    def entropy_from_counts(counts: np.ndarray) -> float:
        counts = np.asarray(counts, dtype=float)
        total = counts.sum()
        if total <= 0:
            return np.nan
        p = counts / total
        p = p[p > 0]
        return float(-np.sum(p * np.log2(p)))


    def compute_jsd_metrics_8(g: pd.DataFrame) -> dict:
        """Recompute within-paper review disagreement using only 8 high-coverage aspects."""
        valid = g[g[ASPECT_TOTAL_8] > 0].copy()
        n_reviews_total = len(g)
        n_reviews_valid = len(valid)

        result = {
            "n_reviews_8aspect_total": n_reviews_total,
            N_REVIEWS_VALID_8: n_reviews_valid,
            JSD_COL_8: np.nan,
            MAX_JSD_COL_8: np.nan,
            CENTROID_JSD_COL_8: np.nan,
            "review_aspect_entropy_mean_8aspect": np.nan,
            "review_aspect_entropy_sd_8aspect": np.nan,
        }

        if n_reviews_valid == 0:
            return result

        counts = valid[ASPECT_CLASSES].astype(float).values
        row_sums = counts.sum(axis=1, keepdims=True)
        probs = counts / row_sums

        entropies = [entropy_from_counts(row) for row in counts]
        result["review_aspect_entropy_mean_8aspect"] = float(np.nanmean(entropies))
        result["review_aspect_entropy_sd_8aspect"] = float(np.nanstd(entropies))

        if n_reviews_valid < 2:
            return result

        pairwise = []
        for i, j in combinations(range(n_reviews_valid), 2):
            pairwise.append(float(jensenshannon(probs[i], probs[j])))

        centroid = probs.mean(axis=0)
        centroid = centroid / centroid.sum()
        centroid_dists = [float(jensenshannon(p, centroid)) for p in probs]

        result[JSD_COL_8] = float(np.mean(pairwise))
        result[MAX_JSD_COL_8] = float(np.max(pairwise))
        result[CENTROID_JSD_COL_8] = float(np.mean(centroid_dists))
        return result


    def sample_count(df: pd.DataFrame, label: str, unit: str = "paper") -> dict:
        row = {"sample": label, "unit": unit}
        if "paper_code" in df.columns:
            row["n_papers"] = df["paper_code"].nunique()
        if "review_id" in df.columns:
            row["n_reviews"] = df["review_id"].nunique()
        elif "n_reviews" in df.columns:
            row["n_reviews"] = int(pd.to_numeric(df["n_reviews"], errors="coerce").fillna(0).sum())
        elif "n_reviews_8aspect_total" in df.columns:
            row["n_reviews"] = int(pd.to_numeric(df["n_reviews_8aspect_total"], errors="coerce").fillna(0).sum())
        return row


    # =========================================================
    # 2. Load strict 12-aspect samples and construct 8-aspect samples
    # =========================================================

    paper_main_12 = read_required(STRICT_SAMPLES / "main_strict_paper_jsd_no_large50.parquet")
    review_main_12 = read_required(STRICT_SAMPLES / "main_strict_review_aligned.parquet")

    # Ensure aspect columns are numeric.
    for cls in ALL_ASPECT_CLASSES:
        if cls in review_main_12.columns:
            review_main_12[cls] = pd.to_numeric(review_main_12[cls], errors="coerce").fillna(0)
        else:
            raise KeyError(f"Missing aspect column in review sample: {cls}")

    review_main_12[ASPECT_TOTAL_8] = review_main_12[ASPECT_CLASSES].sum(axis=1)
    review_main_12["has_8aspect_mention"] = review_main_12[ASPECT_TOTAL_8] > 0

    # Recompute paper-level 8-aspect counts and JSD.
    paper_rows = []
    for paper_code, g in review_main_12.groupby("paper_code"):
        row = {"paper_code": paper_code}
        class_counts = g[ASPECT_CLASSES].sum()
        for cls in ASPECT_CLASSES:
            row[cls] = float(class_counts[cls])
            row[f"{cls}__ratio_8aspect"] = float(class_counts[cls] / class_counts.sum()) if class_counts.sum() > 0 else np.nan
        row[ASPECT_TOTAL_8] = float(class_counts.sum())
        row.update(compute_jsd_metrics_8(g))
        paper_rows.append(row)

    paper_aspect_8 = pd.DataFrame(paper_rows)

    # Merge selected background variables from the strict paper sample.
    bg_cols = [
        "paper_code",
        COUNTRY_COL,
        GENDER_COL,
        "publication_year",
        "subject_primary",
        "n_authors",
        "log_n_authors",
        "n_reviews",
        "n_reviews_valid_for_jsd",
        "aspect_total_count",
        "mean_pairwise_jsd",
        "centroid_jsd",
        "max_pairwise_jsd",
    ]
    bg_cols = [c for c in bg_cols if c in paper_main_12.columns]
    paper_aspect_8 = paper_aspect_8.merge(paper_main_12[bg_cols], on="paper_code", how="left")
    paper_aspect_8 = prepare_categorical_and_numeric(paper_aspect_8)

    # Main 8-aspect JSD-computable paper sample.
    paper_main_8 = paper_aspect_8[
        paper_aspect_8[JSD_COL_8].notna()
        & (paper_aspect_8[N_REVIEWS_VALID_8] >= 2)
    ].copy()

    paper_country_8 = paper_main_8[paper_main_8[COUNTRY_COL] != "Unknown"].copy()
    paper_gender_8 = paper_main_8[paper_main_8[GENDER_COL].isin(["male", "female"])].copy()
    paper_combined_8 = paper_main_8[
        (paper_main_8[COUNTRY_COL] != "Unknown")
        & (paper_main_8[GENDER_COL].isin(["male", "female"]))
    ].copy()

    # Review-level 8-aspect model sample: only reviews with at least one of the 8 aspects.
    review_main_8_all = review_main_12[review_main_12["paper_code"].isin(paper_main_8["paper_code"])].copy()
    review_main_8_valid = review_main_8_all[review_main_8_all[ASPECT_TOTAL_8] > 0].copy()

    # Add/overwrite background fields from 8-aspect paper sample.
    merge_cols = [
        "paper_code",
        COUNTRY_COL,
        GENDER_COL,
        "publication_year",
        "subject_primary",
        "n_authors",
        "log_n_authors",
        ASPECT_TOTAL_8,
        N_REVIEWS_VALID_8,
        JSD_COL_8,
    ]
    merge_cols = [c for c in merge_cols if c in paper_main_8.columns]
    drop_cols = [c for c in merge_cols if c != "paper_code" and c in review_main_8_valid.columns]
    review_main_8_valid = review_main_8_valid.drop(columns=drop_cols, errors="ignore")
    review_main_8_valid = review_main_8_valid.merge(paper_main_8[merge_cols], on="paper_code", how="left")
    review_main_8_valid[ASPECT_TOTAL_8] = review_main_8_valid[ASPECT_CLASSES].sum(axis=1)
    review_main_8_valid = prepare_categorical_and_numeric(review_main_8_valid)

    review_country_8 = review_main_8_valid[review_main_8_valid[COUNTRY_COL] != "Unknown"].copy()
    review_gender_8 = review_main_8_valid[review_main_8_valid[GENDER_COL].isin(["male", "female"])].copy()
    review_combined_8 = review_main_8_valid[
        (review_main_8_valid[COUNTRY_COL] != "Unknown")
        & (review_main_8_valid[GENDER_COL].isin(["male", "female"]))
    ].copy()

    # Save derived 8-aspect samples.
    save_table(paper_aspect_8, OUT_SAMPLES / "paper_aspect_disagreement_8_high_coverage.parquet")
    save_table(paper_main_8, OUT_SAMPLES / "main_strict_paper_jsd_8_high_coverage.parquet")
    save_table(review_main_8_valid, OUT_SAMPLES / "main_strict_review_8_high_coverage_valid.parquet")

    sample_summary = pd.DataFrame([
        sample_count(paper_main_12, "12-aspect strict paper sample retained from previous analysis", "paper"),
        sample_count(review_main_12, "12-aspect strict review sample retained from previous analysis", "review"),
        sample_count(paper_main_8, "8-aspect JSD-computable paper sample", "paper"),
        sample_count(review_main_8_valid, "8-aspect valid review sample", "review"),
        sample_count(paper_country_8, "8-aspect country paper sample", "paper"),
        sample_count(review_country_8, "8-aspect country review sample", "review"),
        sample_count(paper_gender_8, "8-aspect gender paper sample", "paper"),
        sample_count(review_gender_8, "8-aspect gender review sample", "review"),
        sample_count(paper_combined_8, "8-aspect country + gender paper sample", "paper"),
        sample_count(review_combined_8, "8-aspect country + gender review sample", "review"),
    ])
    save_table(sample_summary, OUT_TABLES / "descriptive_sample_summary_8_high_coverage.csv")
    print(sample_summary)


    # =========================================================
    # 3. Descriptive CSV tables for 8-aspect follow-up
    # =========================================================

    def summarize_group_size_paper(df: pd.DataFrame, group_col: str, prefix: str) -> pd.DataFrame:
        tab = (
            df.assign(_group=df[group_col].apply(clean_value))
            .groupby("_group", dropna=False)
            .agg(
                n_papers=("paper_code", "nunique"),
                n_reviews=("n_reviews_8aspect_total", "sum"),
                n_reviews_valid_8=(N_REVIEWS_VALID_8, "sum"),
            )
            .reset_index()
            .rename(columns={"_group": group_col})
            .sort_values("n_papers", ascending=False)
        )
        tab["paper_share"] = tab["n_papers"] / tab["n_papers"].sum()
        save_table(tab, OUT_TABLES / f"descriptive_group_size_paper_8aspect__{prefix}.csv")
        return tab


    def summarize_group_size_review(df: pd.DataFrame, group_col: str, prefix: str) -> pd.DataFrame:
        tab = (
            df.assign(_group=df[group_col].apply(clean_value))
            .groupby("_group", dropna=False)
            .agg(
                n_papers=("paper_code", "nunique"),
                n_reviews=("review_id", "nunique"),
            )
            .reset_index()
            .rename(columns={"_group": group_col})
            .sort_values("n_reviews", ascending=False)
        )
        tab["review_share"] = tab["n_reviews"] / tab["n_reviews"].sum()
        save_table(tab, OUT_TABLES / f"descriptive_group_size_review_8aspect__{prefix}.csv")
        return tab


    def summarize_jsd_8(df: pd.DataFrame, group_col: str | None = None, prefix: str = "overall") -> pd.DataFrame:
        if group_col is None:
            vals = df[JSD_COL_8].dropna()
            out = pd.DataFrame([{
                "group_col": "overall",
                "group": "overall",
                "n_papers": df["paper_code"].nunique(),
                "n_reviews_total": int(df["n_reviews_8aspect_total"].sum()),
                "n_reviews_valid_8": int(df[N_REVIEWS_VALID_8].sum()),
                "mean_jsd_8": vals.mean(),
                "median_jsd_8": vals.median(),
                "sd_jsd_8": vals.std(),
                "q25_jsd_8": vals.quantile(0.25),
                "q75_jsd_8": vals.quantile(0.75),
            }])
        else:
            rows = []
            for group, g in df.assign(_group=df[group_col].apply(clean_value)).groupby("_group"):
                vals = g[JSD_COL_8].dropna()
                rows.append({
                    "group_col": group_col,
                    "group": group,
                    "n_papers": g["paper_code"].nunique(),
                    "n_reviews_total": int(g["n_reviews_8aspect_total"].sum()),
                    "n_reviews_valid_8": int(g[N_REVIEWS_VALID_8].sum()),
                    "mean_jsd_8": vals.mean(),
                    "median_jsd_8": vals.median(),
                    "sd_jsd_8": vals.std(),
                    "q25_jsd_8": vals.quantile(0.25),
                    "q75_jsd_8": vals.quantile(0.75),
                })
            out = pd.DataFrame(rows).sort_values("n_papers", ascending=False)
        save_table(out, OUT_TABLES / f"descriptive_jsd_8aspect__{prefix}.csv")
        return out


    def aspect_overall_distribution_8(df: pd.DataFrame, prefix: str) -> pd.DataFrame:
        total = df[ASPECT_CLASSES].sum().sum()
        n_reviews = df["review_id"].nunique()
        n_papers = df["paper_code"].nunique()
        rows = []
        for cls in ASPECT_CLASSES:
            cnt = float(df[cls].sum())
            review_cover = int((df[cls] > 0).sum())
            paper_cover = int(df.loc[df[cls] > 0, "paper_code"].nunique())
            rows.append({
                "aspect_class": cls,
                "aspect_label": ASPECT_SHORT.get(cls, cls),
                "count_8aspect": cnt,
                "share_among_8_aspects": cnt / total if total > 0 else np.nan,
                "n_reviews_cover": review_cover,
                "review_cover_ratio_in_8_valid_reviews": review_cover / n_reviews if n_reviews > 0 else np.nan,
                "n_papers_cover": paper_cover,
                "paper_cover_ratio_in_8_papers": paper_cover / n_papers if n_papers > 0 else np.nan,
                "n_reviews": n_reviews,
                "n_papers": n_papers,
            })
        out = pd.DataFrame(rows).sort_values("share_among_8_aspects", ascending=False)
        save_table(out, OUT_TABLES / f"descriptive_aspect_overall_8aspect__{prefix}.csv")
        return out

    summarize_group_size_paper(paper_main_8, COUNTRY_COL, "country_all_including_unknown")
    summarize_group_size_paper(paper_main_8, GENDER_COL, "gender_all")
    summarize_group_size_review(review_main_8_valid, COUNTRY_COL, "country_all_including_unknown")
    summarize_group_size_review(review_main_8_valid, GENDER_COL, "gender_all")
    summarize_jsd_8(paper_main_8, prefix="overall")
    summarize_jsd_8(paper_country_8, COUNTRY_COL, "country_known")
    summarize_jsd_8(paper_gender_8, GENDER_COL, "gender_known")
    aspect_overall_distribution_8(review_main_8_valid, "overall")
    aspect_overall_distribution_8(review_country_8, "country_known")
    aspect_overall_distribution_8(review_gender_8, "gender_known")

    # =========================================================
    # 4. Aspect-distribution association tests, 8 aspects
    # =========================================================

    def build_aspect_contingency(df: pd.DataFrame, group_col: str, exclude_groups: Iterable[str] | None = None) -> pd.DataFrame:
        temp = df.copy()
        temp[group_col] = temp[group_col].apply(clean_value)
        if exclude_groups is not None:
            temp = temp[~temp[group_col].isin(set(exclude_groups))].copy()
        contingency = temp.groupby(group_col)[ASPECT_CLASSES].sum().astype(float)
        contingency = contingency.loc[contingency.sum(axis=1) > 0]
        return contingency


    def cramers_v_from_chi2(chi2: float, n: float, r: int, c: int) -> float:
        denom = n * min(r - 1, c - 1)
        return float(np.sqrt(chi2 / denom)) if denom > 0 else np.nan


    def adjusted_standardized_residuals(observed: np.ndarray, expected: np.ndarray) -> np.ndarray:
        observed = np.asarray(observed, dtype=float)
        expected = np.asarray(expected, dtype=float)
        total = observed.sum()
        row_prop = observed.sum(axis=1, keepdims=True) / total
        col_prop = observed.sum(axis=0, keepdims=True) / total
        denom = np.sqrt(expected * (1 - row_prop) * (1 - col_prop))
        with np.errstate(divide="ignore", invalid="ignore"):
            residuals = (observed - expected) / denom
        return residuals


    def chi_square_aspect_test(df: pd.DataFrame, group_col: str, prefix: str, exclude_groups: Iterable[str] | None = None) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
        contingency = build_aspect_contingency(df, group_col, exclude_groups=exclude_groups)
        chi2, p, dof, expected = chi2_contingency(contingency.values)
        n = contingency.values.sum()
        v = cramers_v_from_chi2(chi2, n, contingency.shape[0], contingency.shape[1])

        summary = pd.DataFrame([{
            "group_col": group_col,
            "n_groups": contingency.shape[0],
            "n_aspects": contingency.shape[1],
            "n_aspect_mentions_8": n,
            "chi2": chi2,
            "dof": dof,
            "p_value": p,
            "cramers_v": v,
        }])
        expected_df = pd.DataFrame(expected, index=contingency.index, columns=contingency.columns)
        resid_df = pd.DataFrame(
            adjusted_standardized_residuals(contingency.values, expected),
            index=contingency.index,
            columns=contingency.columns,
        )

        save_table(summary, OUT_TABLES / f"test_chi_square_aspect_8aspect__{prefix}.csv")
        contingency.to_csv(OUT_TABLES / f"test_chi_square_aspect_8aspect__{prefix}__observed.csv", encoding="utf-8-sig")
        expected_df.to_csv(OUT_TABLES / f"test_chi_square_aspect_8aspect__{prefix}__expected.csv", encoding="utf-8-sig")
        resid_df.to_csv(OUT_TABLES / f"test_chi_square_aspect_8aspect__{prefix}__adjusted_residuals.csv", encoding="utf-8-sig")
        return summary, contingency, resid_df

    chi_square_aspect_test(review_country_8, COUNTRY_COL, "country", exclude_groups=["Unknown"])
    chi_square_aspect_test(review_gender_8, GENDER_COL, "gender", exclude_groups=["Unknown"])


    # =========================================================
    # 5. Review-level aspect-ratio OLS models, 8 aspects
    # =========================================================

    def add_review_model_features_8(df: pd.DataFrame) -> pd.DataFrame:
        out = df.copy()
        out[ASPECT_TOTAL_8] = out[ASPECT_CLASSES].sum(axis=1)
        for cls in ASPECT_CLASSES:
            y = aspect_ratio_col(cls)
            out[y] = np.where(out[ASPECT_TOTAL_8] > 0, out[cls] / out[ASPECT_TOTAL_8], np.nan)
        if "review_word_count" in out.columns:
            out["log_review_word_count"] = safe_log1p(out["review_word_count"])
        else:
            out["log_review_word_count"] = 0.0
        return out


    def fit_review_aspect_ols_8(df: pd.DataFrame, predictor_type: str, formula_rhs: str, prefix: str) -> pd.DataFrame:
        rows = []
        temp = add_review_model_features_8(df)
        base_cols = [
            "paper_code",
            COUNTRY_COL,
            GENDER_COL,
            "publication_year",
            "subject_primary",
            "log_review_word_count",
            "log_n_authors",
        ]
        base_cols = [c for c in base_cols if c in temp.columns]

        for cls in ASPECT_CLASSES:
            y = aspect_ratio_col(cls)
            model_cols = [y] + base_cols
            model_df = temp[model_cols].copy()
            model_df = coerce_for_statsmodels(
                model_df,
                numeric_cols=[y, "publication_year", "log_review_word_count", "log_n_authors"],
                categorical_cols=[COUNTRY_COL, GENDER_COL, "subject_primary"],
                group_col="paper_code",
            )
            model_df = model_df.dropna(subset=[y, "publication_year", "log_review_word_count", "log_n_authors"]).copy()
            if model_df.empty:
                continue

            formula = f"Q('{y}') ~ {formula_rhs}"
            try:
                model = smf.ols(formula, data=model_df).fit(
                    cov_type="cluster",
                    cov_kwds={"groups": model_df["paper_code"]},
                )
            except Exception as e:
                print(f"Review model failed: {cls} | {prefix} | {e}")
                continue

            conf_int = model.conf_int()
            for term, est in model.params.items():
                if term == "Intercept":
                    continue
                conf = conf_int.loc[term]
                rows.append({
                    "aspect_set": "8_high_coverage",
                    "aspect_class": cls,
                    "predictor_type": predictor_type,
                    "term": term,
                    "estimate": est,
                    "conf_low": conf[0],
                    "conf_high": conf[1],
                    "p_value": model.pvalues.get(term, np.nan),
                    "n_reviews": int(model.nobs),
                    "n_papers": model_df["paper_code"].nunique(),
                    "model": prefix,
                })

        out = pd.DataFrame(rows)
        if len(out) > 0:
            out["p_adj_bh"] = multipletests(out["p_value"].fillna(1), method="fdr_bh")[1]
        save_table(out, OUT_TABLES / f"model_review_aspect_ratio_ols_8aspect__{prefix}.csv")
        return out

    base_review_controls = "publication_year + C(subject_primary) + log_review_word_count + log_n_authors"
    review_country_models_8 = fit_review_aspect_ols_8(
        review_country_8,
        predictor_type="first_author_country",
        formula_rhs=f"C({COUNTRY_COL}, Treatment(reference='{COUNTRY_REFERENCE}')) + {base_review_controls}",
        prefix="country",
    )
    review_gender_models_8 = fit_review_aspect_ols_8(
        review_gender_8,
        predictor_type="first_author_gender",
        formula_rhs=f"C({GENDER_COL}, Treatment(reference='{GENDER_REFERENCE}')) + {base_review_controls}",
        prefix="gender",
    )
    review_combined_models_8 = fit_review_aspect_ols_8(
        review_combined_8,
        predictor_type="country_and_gender",
        formula_rhs=f"C({COUNTRY_COL}, Treatment(reference='{COUNTRY_REFERENCE}')) + C({GENDER_COL}, Treatment(reference='{GENDER_REFERENCE}')) + {base_review_controls}",
        prefix="country_gender",
    )


    # =========================================================
    # 6. Paper-level 8-aspect JSD OLS models
    # =========================================================

    def fit_jsd_model_8(df: pd.DataFrame, predictor_type: str, formula_rhs: str, prefix: str, y_col: str = JSD_COL_8) -> pd.DataFrame:
        needed = [
            "paper_code",
            y_col,
            "publication_year",
            "subject_primary",
            N_REVIEWS_VALID_8,
            "log_aspect_total_count_8aspect",
            "log_n_authors",
        ]
        if COUNTRY_COL in formula_rhs:
            needed.append(COUNTRY_COL)
        if GENDER_COL in formula_rhs:
            needed.append(GENDER_COL)
        needed = [c for c in dict.fromkeys(needed) if c in df.columns]

        model_df = df[needed].copy()
        model_df = coerce_for_statsmodels(
            model_df,
            numeric_cols=[y_col, "publication_year", N_REVIEWS_VALID_8, "log_aspect_total_count_8aspect", "log_n_authors"],
            categorical_cols=[COUNTRY_COL, GENDER_COL, "subject_primary"],
            group_col="paper_code",
        )
        model_df = model_df.dropna(subset=[y_col, "publication_year", N_REVIEWS_VALID_8, "log_aspect_total_count_8aspect", "log_n_authors"]).copy()
        formula = f"{y_col} ~ {formula_rhs}"
        model = smf.ols(formula, data=model_df).fit(cov_type="HC3")

        rows = []
        conf_int = model.conf_int()
        for term, est in model.params.items():
            if term == "Intercept":
                continue
            conf = conf_int.loc[term]
            rows.append({
                "aspect_set": "8_high_coverage",
                "dependent_var": y_col,
                "predictor_type": predictor_type,
                "term": term,
                "estimate": est,
                "conf_low": conf[0],
                "conf_high": conf[1],
                "p_value": model.pvalues.get(term, np.nan),
                "n_papers": int(model.nobs),
                "r_squared": model.rsquared,
                "adj_r_squared": model.rsquared_adj,
                "model": prefix,
            })

        out = pd.DataFrame(rows)
        if len(out) > 0:
            out["p_adj_bh"] = multipletests(out["p_value"].fillna(1), method="fdr_bh")[1]
        save_table(out, OUT_TABLES / f"model_paper_jsd_ols_8aspect__{prefix}.csv")
        return out

    base_jsd_controls = f"publication_year + C(subject_primary) + {N_REVIEWS_VALID_8} + log_aspect_total_count_8aspect + log_n_authors"

    jsd_country_8 = fit_jsd_model_8(
        paper_country_8,
        predictor_type="first_author_country",
        formula_rhs=f"C({COUNTRY_COL}, Treatment(reference='{COUNTRY_REFERENCE}')) + {base_jsd_controls}",
        prefix="country",
    )
    jsd_gender_8 = fit_jsd_model_8(
        paper_gender_8,
        predictor_type="first_author_gender",
        formula_rhs=f"C({GENDER_COL}, Treatment(reference='{GENDER_REFERENCE}')) + {base_jsd_controls}",
        prefix="gender",
    )
    jsd_combined_8 = fit_jsd_model_8(
        paper_combined_8,
        predictor_type="country_and_gender",
        formula_rhs=f"C({COUNTRY_COL}, Treatment(reference='{COUNTRY_REFERENCE}')) + C({GENDER_COL}, Treatment(reference='{GENDER_REFERENCE}')) + {base_jsd_controls}",
        prefix="country_gender",
    )
    jsd_all_8 = pd.concat([jsd_country_8, jsd_gender_8, jsd_combined_8], ignore_index=True)
    save_table(jsd_all_8, OUT_TABLES / "model_paper_jsd_ols_8aspect__all_models.csv")


    # =========================================================
    # 7. Per-aspect disagreement, 8 aspects
    # =========================================================

    def mean_pairwise_abs(values: np.ndarray) -> float:
        values = np.asarray(values, dtype=float)
        values = values[np.isfinite(values)]
        n = len(values)
        if n < 2:
            return np.nan
        values = np.sort(values)
        weights = 2 * np.arange(n) - n + 1
        pair_sum = np.sum(weights * values)
        return float(pair_sum / (n * (n - 1) / 2))


    def compute_per_aspect_disagreement_8(review_df: pd.DataFrame) -> pd.DataFrame:
        temp = review_df.copy()
        temp[ASPECT_TOTAL_8] = temp[ASPECT_CLASSES].sum(axis=1)
        temp = temp[temp[ASPECT_TOTAL_8] > 0].copy()

        for cls in ASPECT_CLASSES:
            temp[share_col(cls)] = temp[cls] / temp[ASPECT_TOTAL_8]

        rows = []
        for paper_code, g in temp.groupby("paper_code"):
            row = {"paper_code": paper_code, "n_reviews_valid_8_for_mapd": g["review_id"].nunique()}
            for cls in ASPECT_CLASSES:
                row[f"mapd8__{cls}"] = mean_pairwise_abs(g[share_col(cls)].values)
            rows.append(row)

        out = pd.DataFrame(rows)
        merge_cols = [
            "paper_code",
            COUNTRY_COL,
            GENDER_COL,
            "publication_year",
            "subject_primary",
            "n_authors",
            "log_n_authors",
            N_REVIEWS_VALID_8,
            ASPECT_TOTAL_8,
            "log_aspect_total_count_8aspect",
            JSD_COL_8,
        ]
        merge_cols = [c for c in merge_cols if c in paper_main_8.columns]
        out = out.merge(paper_main_8[merge_cols], on="paper_code", how="left")
        return out

    per_aspect_8 = compute_per_aspect_disagreement_8(review_main_8_valid)
    save_table(per_aspect_8, OUT_SAMPLES / "main_strict_paper_per_aspect_disagreement_8_high_coverage.parquet")

    per_aspect_summary_rows = []
    for cls in ASPECT_CLASSES:
        vals = per_aspect_8[f"mapd8__{cls}"].dropna()
        per_aspect_summary_rows.append({
            "aspect_set": "8_high_coverage",
            "aspect_class": cls,
            "aspect_label": ASPECT_SHORT.get(cls, cls),
            "n_papers": len(vals),
            "mean_mapd": vals.mean(),
            "median_mapd": vals.median(),
            "sd_mapd": vals.std(),
            "q25_mapd": vals.quantile(0.25),
            "q75_mapd": vals.quantile(0.75),
        })
    per_aspect_summary_8 = pd.DataFrame(per_aspect_summary_rows).sort_values("mean_mapd", ascending=False)
    save_table(per_aspect_summary_8, OUT_TABLES / "descriptive_per_aspect_disagreement_8aspect.csv")

    # =========================================================
    # 8. Final figures, English text, no titles
    # =========================================================

    def aspect_distribution_by_group_8(df: pd.DataFrame, group_col: str) -> pd.DataFrame:
        rows = []
        temp = df.copy()
        temp[group_col] = temp[group_col].apply(clean_value)
        for group, g in temp.groupby(group_col):
            total = g[ASPECT_CLASSES].sum().sum()
            if total <= 0:
                continue
            for cls in ASPECT_CLASSES:
                rows.append({
                    "group": group,
                    "aspect_class": cls,
                    "aspect_label": ASPECT_SHORT.get(cls, cls),
                    "aspect_ratio_8": g[cls].sum() / total,
                    "n_papers": g["paper_code"].nunique(),
                    "n_reviews": g["review_id"].nunique(),
                })
        return pd.DataFrame(rows)


    def make_deviation_matrix_8(df: pd.DataFrame, group_col: str, groups_to_plot: list[str] | None = None) -> pd.DataFrame:
        dist = aspect_distribution_by_group_8(df, group_col)
        mat = dist.pivot(index="group", columns="aspect_class", values="aspect_ratio_8")
        mat = mat[[c for c in ASPECT_CLASSES if c in mat.columns]]

        overall = df[ASPECT_CLASSES].sum()
        overall = overall / overall.sum()
        for cls in mat.columns:
            mat[cls] = mat[cls] - overall[cls]
        mat = mat * 100

        if groups_to_plot is not None:
            mat = mat.loc[[g for g in groups_to_plot if g in mat.index]]
        return mat


    def plot_heatmap(mat: pd.DataFrame, output_name: str, cbar_label: str, annotate: bool = True, fmt: str = ".2f", figsize: tuple[float, float] | None = None) -> None:
        data = mat.copy()
        data.columns = [ASPECT_SHORT.get(c, c) for c in data.columns]
        if figsize is None:
            figsize = (10.8, max(2.8, 0.42 * data.shape[0] + 1.6))

        fig, ax = plt.subplots(figsize=figsize)
        values = data.values
        vmax = np.nanmax(np.abs(values)) if np.isfinite(values).any() else 1
        norm = TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)
        im = ax.imshow(values, aspect="auto", cmap=DIVERGING_CMAP, norm=norm)

        ax.set_xticks(np.arange(data.shape[1]))
        ax.set_xticklabels(data.columns, rotation=35, ha="right")
        ax.set_yticks(np.arange(data.shape[0]))
        ax.set_yticklabels(data.index)
        ax.set_xlabel("")
        ax.set_ylabel("")

        ax.set_xticks(np.arange(-0.5, data.shape[1], 1), minor=True)
        ax.set_yticks(np.arange(-0.5, data.shape[0], 1), minor=True)
        ax.grid(which="minor", color="white", linestyle="-", linewidth=0.7)
        ax.tick_params(which="minor", bottom=False, left=False)

        if annotate:
            for i in range(data.shape[0]):
                for j in range(data.shape[1]):
                    val = values[i, j]
                    if np.isfinite(val):
                        ax.text(j, i, format(val, fmt), ha="center", va="center", fontsize=7)

        cbar = fig.colorbar(im, ax=ax, fraction=0.025, pad=0.02)
        cbar.set_label(cbar_label)
        save_fig(output_name)
        plt.close(fig)


    def extract_group_from_term(term: str) -> str:
        match = re.search(r"\[T\.(.*?)\]", term)
        return match.group(1) if match else term


    def plot_coef_table(
        df: pd.DataFrame,
        output_name: str,
        xlabel: str,
        label_col: str = "label",
        figsize: tuple[float, float] | None = None,
        sort_by: str = "estimate",
        annotate_values: bool = True,
    ) -> None:
        plot_df = df.copy().sort_values(sort_by, ascending=True)
        if figsize is None:
            figsize = (7.2, max(2.8, 0.36 * len(plot_df) + 1.0))

        fig, ax = plt.subplots(figsize=figsize)
        y = np.arange(len(plot_df))
        xerr = np.vstack([
            plot_df["estimate"].values - plot_df["conf_low"].values,
            plot_df["conf_high"].values - plot_df["estimate"].values,
        ])
        ax.errorbar(plot_df["estimate"], y, xerr=xerr, fmt="o", capsize=3, markersize=4, linewidth=1)
        ax.axvline(0, linestyle="--", linewidth=1, color="gray")
        ax.set_yticks(y)
        ax.set_yticklabels(plot_df[label_col])
        ax.set_xlabel(xlabel)
        ax.set_ylabel("")

        if annotate_values and len(plot_df) > 0:
            x_min = np.nanmin(plot_df["conf_low"].values)
            x_max = np.nanmax(plot_df["conf_high"].values)
            pad = (x_max - x_min) * 0.02 if x_max > x_min else 0.002
            for i, row in plot_df.reset_index(drop=True).iterrows():
                x_text = row["conf_high"] + pad if row["estimate"] >= 0 else row["conf_low"] - pad
                ha = "left" if row["estimate"] >= 0 else "right"
                ax.text(x_text, i, f"{row['estimate']:.4f}", va="center", ha=ha, fontsize=8)
            ax.set_xlim(x_min - 5 * pad, x_max + 8 * pad)

        save_fig(output_name)
        plt.close(fig)

    # Country aspect deviation heatmap. Hide Other/Unknown in the figure only.
    country_fig_df = review_country_8[~review_country_8[COUNTRY_COL].isin(["Other", "Unknown"])].copy()
    country_order = (
        country_fig_df.groupby(COUNTRY_COL)["paper_code"].nunique()
        .sort_values(ascending=False)
        .index.tolist()
    )
    country_dev_8 = make_deviation_matrix_8(country_fig_df, COUNTRY_COL, country_order)
    country_dev_8.to_csv(OUT_TABLES / "figure_data_country_aspect_deviation_pp_8aspect.csv", encoding="utf-8-sig")
    plot_heatmap(
        country_dev_8,
        output_name="fig_country_aspect_deviation_heatmap_8aspect.png",
        cbar_label="Difference from overall share (pp)",
        annotate=True,
        fmt=".2f",
        figsize=(10.8, 6.0),
    )

    # Gender aspect deviation heatmap.
    gender_order = [g for g in ["male", "female"] if g in review_gender_8[GENDER_COL].unique()]
    gender_dev_8 = make_deviation_matrix_8(review_gender_8, GENDER_COL, gender_order)
    gender_dev_8.to_csv(OUT_TABLES / "figure_data_gender_aspect_deviation_pp_8aspect.csv", encoding="utf-8-sig")
    plot_heatmap(
        gender_dev_8,
        output_name="fig_gender_aspect_deviation_heatmap_8aspect.png",
        cbar_label="Difference from overall share (pp)",
        annotate=True,
        fmt=".2f",
        figsize=(10.8, 2.3),
    )

    # JSD country coefficients.
    jsd_country_plot_8 = jsd_country_8[jsd_country_8["term"].str.contains(f"C\\({COUNTRY_COL}", regex=True)].copy()
    jsd_country_plot_8["label"] = jsd_country_plot_8["term"].apply(extract_group_from_term)
    jsd_country_plot_8 = jsd_country_plot_8[~jsd_country_plot_8["label"].isin(["Other", "Unknown"])].copy()
    jsd_country_plot_8.to_csv(OUT_TABLES / "figure_data_jsd_country_coefficients_8aspect.csv", index=False, encoding="utf-8-sig")
    plot_coef_table(
        jsd_country_plot_8,
        output_name="fig_jsd_country_coefficients_8aspect.png",
        xlabel=f"Coefficient on 8-aspect mean pairwise JSD (reference: {COUNTRY_REFERENCE})",
        figsize=(7.5, max(4.8, 0.36 * len(jsd_country_plot_8) + 1.2)),
        annotate_values=True,
    )

    # JSD gender coefficients.
    jsd_gender_plot_8 = jsd_gender_8[jsd_gender_8["term"].str.contains(f"C\\({GENDER_COL}", regex=True)].copy()
    jsd_gender_plot_8["label"] = jsd_gender_plot_8["term"].apply(extract_group_from_term)
    jsd_gender_plot_8["label"] = jsd_gender_plot_8["label"].replace({"female": "Female first author"})
    jsd_gender_plot_8.to_csv(OUT_TABLES / "figure_data_jsd_gender_coefficients_8aspect.csv", index=False, encoding="utf-8-sig")
    plot_coef_table(
        jsd_gender_plot_8,
        output_name="fig_jsd_gender_coefficients_8aspect.png",
        xlabel=f"Coefficient on 8-aspect mean pairwise JSD (reference: {GENDER_REFERENCE})",
        figsize=(7.0, 2.2),
        annotate_values=True,
    )

    # 8-aspect JSD distribution.
    jsd = paper_main_8[JSD_COL_8].dropna()
    fig, ax = plt.subplots(figsize=(7.2, 4.3))
    ax.hist(jsd, bins=40, edgecolor="white", linewidth=0.4)
    ax.axvline(jsd.mean(), linestyle="--", linewidth=1.3, label=f"Mean = {jsd.mean():.3f}")
    ax.axvline(jsd.median(), linestyle=":", linewidth=1.6, label=f"Median = {jsd.median():.3f}")
    ax.set_xlabel("8-aspect mean pairwise JSD")
    ax.set_ylabel("Number of papers")
    ax.legend(frameon=False)
    save_fig("fig_jsd_distribution_8aspect.png")
    plt.close(fig)

    # Per-aspect average disagreement.
    plot_mapd = per_aspect_summary_8.sort_values("mean_mapd", ascending=True).copy()
    fig, ax = plt.subplots(figsize=(7.0, 4.4))
    ax.barh(plot_mapd["aspect_label"], plot_mapd["mean_mapd"] * 100)
    ax.set_xlabel("Mean pairwise absolute difference (pp)")
    ax.set_ylabel("")
    for i, v in enumerate(plot_mapd["mean_mapd"] * 100):
        ax.text(v, i, f" {v:.2f}", va="center", fontsize=8)
    save_fig("fig_per_aspect_disagreement_overall_8aspect.png")
    plt.close(fig)

    print("\nDone. Eight-aspect follow-up analysis completed without modifying the 12-aspect descriptive results.")


In [ ]:
run_high_coverage_8()
